# Real-Time Control

Run from `Code/`, with the input datasets in `../Data/`. Requires NumPy, pandas, Matplotlib, openpyxl, Pyomo, HiGHS and IPython. Figures and tables are displayed in the notebook; no images, result files or solver logs are exported.


In [ ]:
import pyomo.environ as pe
import numpy as np
import os
import pandas as pd
from datetime import datetime
from datetime import timedelta
import matplotlib.pyplot as plt
from dataclasses import dataclass
from matplotlib.patches import Patch    

In [ ]:
# ============================================================
# GLOBAL TIME AND LIMIT SETTINGS
# ============================================================
BLOCK_SIZE = 15  # minutes
N_BLOCKS = 96  # 24h * 4 blocks per hour
TOTAL_MINUTES = N_BLOCKS * BLOCK_SIZE

# Política para os blocos incompletos (ON inicial / OFF final):
# "nearest": ambas as durações vêm do episódio OFF -> ON completo mais próximo.
# "observed": ambas as durações são iguais à duração do bloco observado.
# Depois de alterar, executar novamente todas as células para atualizar o RT.
INCOMPLETE_POLICY = "observed"  # Alternar entre "nearest" e "observed".

if INCOMPLETE_POLICY not in {"nearest", "observed"}:
    raise ValueError("INCOMPLETE_POLICY must be 'nearest' or 'observed'.")


# ============================================================
# TIME ZONES
# ============================================================
#
# 0 -> 00:00 - 09:59
# 1 -> 10:00 - 19:29
# 2 -> 19:30 - 22:29
# 3 -> 22:30 - 23:59
#
# start and end are expressed in minutes
# ============================================================

TIME_ZONES = {
    0: {
        "start": 0,
        "end": 600
    },

    1: {
        "start": 600,
        "end": 1170
    },

    2: {
        "start": 1170,
        "end": 1350
    },

    3: {
        "start": 1350,
        "end": 1440
    }
}


# ============================================================
# FIXED POWER LIMITS BY TIMEZONE [W]
# These are service limits, independent of the observed daily peak.
# ============================================================

TIMEZONE_POWER_LIMITS_W = {
    0: 2500.0,
    1: 13000.0,
    2: 16000.0,
    3: 7500.0,
}
TOU_PRICE_EUR_KWH = {
    0: 0.15,   # Off-peak: 00:00 - 10:00
    1: 0.25,   # Shoulder: 10:00 - 19:30
    2: 0.45,   # On-peak: 19:30 - 22:30
    3: 0.15,   # Off-peak: 22:30 - 24:00
}

# Auxiliary functions

In [ ]:
"""Minute-level diagnostics for the two standalone real-time control strategies.

Inputs cover the full day, starting at minute zero. Power is supplied in W;
reported power is in kW. Availability is measured BEFORE optimization.
"""

import numpy as np
import pandas as pd


def analyze_rt_flexibility(
    *,
    X_reference,
    Y_reference,
    X_rt,
    Y_rt,
    force_on,
    force_off,
    prevent_off,
    count_hold_after,
    Type,
    Power_max,
    Y_min,
    dev_names,
    P_lim,
    tolerance_W=1e-6,
):
    """
    Return:

        minute_metrics,
        violations,
        device_summary

    Availability is evaluated before optimization.

    Time flexibility
    ----------------
    A Type-2 device contributes when it is ON immediately before the solver
    and none of force_on, force_off, or prevent_off blocks a new curtailment.

    Power flexibility
    -----------------
    For an active Type-1 device:

        available PF = Pmax * max(Y_before - Ymin, 0)

    Example:

        Y_before = 0.70
        Y_min = 0.30

        available PF = 0.40 * Pmax
    """
    import numpy as np
    import pandas as pd

    X_reference = np.asarray(
        X_reference,
        dtype=float,
    )

    if (
        X_reference.ndim != 2
        or X_reference.shape[0] == 0
    ):
        raise ValueError(
            "X_reference must be a non-empty "
            "(minutes, devices) matrix."
        )

    n_minutes, n_devices = X_reference.shape
    expected_shape = (n_minutes, n_devices)

    def matrix(value, name, binary=False):
        result = np.asarray(value, dtype=float)

        if result.shape != expected_shape:
            raise ValueError(
                f"{name} must have shape "
                f"{expected_shape}; received {result.shape}."
            )

        if not np.isfinite(result).all():
            raise ValueError(
                f"{name} must contain finite values."
            )

        if (
            binary
            and not np.isin(result, [0, 1]).all()
        ):
            raise ValueError(
                f"{name} must contain only 0 and 1."
            )

        return result

    def vector(value, name, size):
        result = np.asarray(value, dtype=float)

        if result.shape != (size,):
            raise ValueError(
                f"{name} must have length {size}."
            )

        if not np.isfinite(result).all():
            raise ValueError(
                f"{name} must contain finite values."
            )

        return result

    X_reference = matrix(
        X_reference,
        "X_reference",
        binary=True,
    )

    Y_reference = matrix(
        Y_reference,
        "Y_reference",
    )

    X_rt = matrix(
        X_rt,
        "X_rt",
        binary=True,
    )

    Y_rt = matrix(
        Y_rt,
        "Y_rt",
    )

    force_on = matrix(
        force_on,
        "force_on",
        binary=True,
    ).astype(bool)

    force_off = matrix(
        force_off,
        "force_off",
        binary=True,
    ).astype(bool)

    prevent_off = matrix(
        prevent_off,
        "prevent_off",
        binary=True,
    ).astype(bool)

    hold_after = matrix(
        count_hold_after,
        "count_hold_after",
    )

    types = vector(
        Type,
        "Type",
        n_devices,
    )

    rated_power = vector(
        Power_max,
        "Power_max",
        n_devices,
    )

    power_limits = vector(
        P_lim,
        "P_lim",
        n_minutes,
    )

    minimum_power = np.asarray(
        Y_min,
        dtype=float,
    )

    device_names = np.asarray(
        dev_names,
        dtype=object,
    )

    if minimum_power.shape != (n_devices,):
        raise ValueError(
            "Y_min must have one value per device."
        )

    if device_names.shape != (n_devices,):
        raise ValueError(
            "dev_names must have one name per device."
        )

    if not np.isin(types, [0, 1, 2]).all():
        raise ValueError(
            "Type must contain only 0, 1, or 2."
        )

    if (rated_power < 0).any():
        raise ValueError(
            "Power_max cannot contain negative values."
        )

    if (hold_after < 0).any():
        raise ValueError(
            "Hold counters cannot be negative."
        )

    if (force_on & force_off).any():
        raise ValueError(
            "A device cannot be force_on and "
            "force_off simultaneously."
        )

    if (force_on & prevent_off).any():
        # Isto não é necessariamente impossível,
        # mas normalmente indica flags redundantes.
        pass

    if not np.isfinite(tolerance_W):
        raise ValueError(
            "tolerance_W must be finite."
        )

    is_tf = types == 2
    is_pf = types == 1

    if not np.isfinite(
        minimum_power[is_pf]
    ).all():
        raise ValueError(
            "Y_min must be finite for Type-1 devices."
        )

    # ========================================================
    # STATE BEFORE OPTIMIZATION
    # ========================================================

    # Estado efetivamente enviado ao modelo.
    x_before = np.where(
        force_on,
        1,
        np.where(
            force_off,
            0,
            X_reference,
        ),
    )

    # Reconstruct Y_before with the same hold sequence
    # used by the RT loops.
    previous_hold = np.vstack([
        np.zeros((1, n_devices)),
        hold_after[:-1],
    ])

    previous_y = np.vstack([
        np.ones((1, n_devices)),
        Y_rt[:-1],
    ])

    active_pf = (
        is_pf[np.newaxis, :]
        & (x_before == 1)
    )
    adjustable_pf = (
        active_pf
        & (previous_hold <= 0)
    )

    y_before = np.where(
        active_pf,
        np.where(
            previous_hold > 0,
            previous_y,
            Y_reference,
        ),
        1.0,
    )

    # ========================================================
    # AVAILABLE TIME FLEXIBILITY
    # ========================================================

    # Downward flexibility that is still available immediately before the
    # solver. A force_off device is already OFF at this point and therefore
    # cannot provide any additional reduction.
    eligible_tf = (
        is_tf[np.newaxis, :]
        & (x_before == 1)
        & ~force_on
        & ~force_off
        & ~prevent_off
    )

    flex_tf_W = (
        eligible_tf
        * rated_power[np.newaxis, :]
    )

    # ========================================================
    # AVAILABLE POWER FLEXIBILITY
    # ========================================================

    flex_pf_W = np.zeros(
        expected_shape,
        dtype=float,
    )

    pf_headroom = np.maximum(
        y_before[:, is_pf]
        - minimum_power[is_pf],
        0.0,
    )

    flex_pf_W[:, is_pf] = np.where(
        adjustable_pf[:, is_pf],
        rated_power[is_pf] * pf_headroom,
        0.0,
    )

    # ========================================================
    # POWER BEFORE AND AFTER OPTIMIZATION
    # ========================================================

    power_before_W = (
        x_before
        * y_before
        * rated_power[np.newaxis, :]
    ).sum(axis=1)

    power_after_W = (
        X_rt
        * Y_rt
        * rated_power[np.newaxis, :]
    ).sum(axis=1)

    available_total_W = (
        flex_tf_W + flex_pf_W
    ).sum(axis=1)

    required_reduction_W = np.maximum(
        power_before_W - power_limits,
        0.0,
    )

    exceedance_W = np.maximum(
        power_after_W - power_limits,
        0.0,
    )

    violation = exceedance_W > tolerance_W

    exceedance_W = np.where(
        violation,
        exceedance_W,
        0.0,
    )

    def device_lists(mask):
        return [
            tuple(device_names[row_mask])
            for row_mask in mask
        ]

    # ========================================================
    # MINUTE-LEVEL RESULTS
    # ========================================================

    metrics = pd.DataFrame({
        "Minute": np.arange(n_minutes),
        "Time": [
            f"{minute // 60:02d}:{minute % 60:02d}"
            for minute in range(n_minutes)
        ],

        "Power before [kW]": (
            power_before_W / 1000
        ),

        "Power after [kW]": (
            power_after_W / 1000
        ),

        "Power limit [kW]": (
            power_limits / 1000
        ),

        "Required reduction [kW]": (
            required_reduction_W / 1000
        ),

        "Violation": violation,

        "Exceedance [kW]": (
            exceedance_W / 1000
        ),

        "Force ON devices": (
            device_lists(force_on)
        ),

        "Force ON count": (
            force_on.sum(axis=1)
        ),

        "Prevent OFF devices": (
            device_lists(prevent_off)
        ),

        "Prevent OFF count": (
            prevent_off.sum(axis=1)
        ),

        "Available TF devices": (
            device_lists(eligible_tf)
        ),

        "Available TF count": (
            eligible_tf.sum(axis=1)
        ),

        "Available TF [kW]": (
            flex_tf_W.sum(axis=1) / 1000
        ),

        "Available PF devices": (
            device_lists(
                flex_pf_W > tolerance_W
            )
        ),

        "Available PF count": (
            (flex_pf_W > tolerance_W).sum(axis=1)
        ),

        "Available PF [kW]": (
            flex_pf_W.sum(axis=1) / 1000
        ),

        "Available total [kW]": (
            available_total_W / 1000
        ),
    })

    violations = (
        metrics.loc[violation]
        .copy()
        .reset_index(drop=True)
    )

    device_summary = pd.DataFrame({
        "Device": device_names[is_tf],

        "Force ON duration [min]": (
            force_on[:, is_tf].sum(axis=0)
        ),

        "Prevent OFF duration [min]": (
            prevent_off[:, is_tf].sum(axis=0)
        ),

        "Available duration [min]": (
            eligible_tf[:, is_tf].sum(axis=0)
        ),
    })

    return metrics, violations, device_summary


def summarize_rt_flexibility(
    scenarios,
    time_start=13 * 60,
    time_finish=15 * 60,
    tolerance_kW=1e-9,
):
    """
    Summarize flexibility availability over a selected interval.

    Metrics
    -------
    Average available flexibility [kW]
        Mean instantaneous downward flexibility available before optimization
        over the complete selected interval.

    Minutes requiring reduction
        Number of minutes in which pre-optimization power exceeds the
        power limit, i.e. Required reduction > 0.

    Minutes with sufficient flexibility
        Among the minutes requiring reduction, number of minutes in which
        available flexibility is at least equal to the required reduction.
    """

    if not scenarios:
        raise ValueError(
            "scenarios cannot be empty."
        )

    if not (
        0 <= time_start
        < time_finish
        <= 24 * 60
    ):
        raise ValueError(
            "Expected "
            "0 <= time_start < time_finish <= 1440."
        )

    required_columns = {
        "Minute",
        "Required reduction [kW]",
        "Available total [kW]",
    }

    rows = []

    for scenario, metrics in scenarios.items():

        if not isinstance(
            metrics,
            pd.DataFrame
        ):
            raise TypeError(
                f"Metrics for {scenario!r} "
                "must be a pandas DataFrame."
            )

        missing = (
            required_columns
            .difference(metrics.columns)
        )

        if missing:
            raise ValueError(
                f"Metrics for {scenario!r} "
                f"are missing columns: "
                f"{sorted(missing)}."
            )

        # ====================================================
        # SELECT INTERVAL
        # ====================================================

        window = metrics.loc[
            (metrics["Minute"] >= time_start)
            & (metrics["Minute"] < time_finish)
        ].copy()

        expected_minutes = (
            time_finish - time_start
        )

        if len(window) != expected_minutes:
            raise ValueError(
                f"Expected {expected_minutes} "
                f"one-minute rows for "
                f"{scenario!r}, "
                f"received {len(window)}."
            )

        if (
            window[
                list(required_columns)
            ]
            .isna()
            .any()
            .any()
        ):
            raise ValueError(
                f"Metrics for {scenario!r} "
                "contain missing values in "
                "the requested interval."
            )

        # ====================================================
        # VALUES
        # ====================================================

        available = (
            window[
                "Available total [kW]"
            ]
            .to_numpy(dtype=float)
        )

        required = (
            window[
                "Required reduction [kW]"
            ]
            .to_numpy(dtype=float)
        )

        # ====================================================
        # 1. AVERAGE AVAILABLE FLEXIBILITY
        # ====================================================

        average_available = float(
            available.mean()
        )

        # ====================================================
        # 2. MINUTES REQUIRING REDUCTION
        # ====================================================

        requiring_reduction = (
            required > tolerance_kW
        )

        minutes_requiring = int(
            requiring_reduction.sum()
        )

        # ====================================================
        # 3. MINUTES WITH SUFFICIENT FLEXIBILITY
        # ====================================================

        sufficient_flexibility = (
            requiring_reduction
            & (
                available
                + tolerance_kW
                >= required
            )
        )

        minutes_sufficient = int(
            sufficient_flexibility.sum()
        )
        
        sufficient_rate = (
            100.0 * minutes_sufficient / minutes_requiring
            if minutes_requiring > 0
            else np.nan
        )
        # ====================================================
        # STORE RESULTS
        # ====================================================

        rows.append({
            "Scenario": scenario,

            "Interval": (
                f"{time_start // 60:02d}:"
                f"{time_start % 60:02d}-"
                f"{time_finish // 60:02d}:"
                f"{time_finish % 60:02d}"
            ),

            "Average available flexibility [kW]":
                average_available,

            "Minutes requiring reduction":
                minutes_requiring,

            "Minutes with sufficient flexibility":
                minutes_sufficient,
            "Sufficient flexibility rate [%]":
                sufficient_rate,
        })

    return pd.DataFrame(rows)


def plot_rt_flexibility(scenarios, time_start=13*60, time_finish=15*60,
                        tick_step=15):
    """Compare pre-optimization available reduction; finish is exclusive.

    scenarios maps display names to the first table from analyze_rt_flexibility.
    Returns (figure, axes). Red one-minute horizontal segments mark observed
    post-control violations.
    """
    import matplotlib.pyplot as plt

    if not scenarios or not 0 <= time_start < time_finish or tick_step <= 0:
        raise ValueError("Supply scenarios, a valid interval, and positive tick_step.")
    fig, axes = plt.subplots(
        len(scenarios), 1, figsize=(14, 3.5 * len(scenarios)),
        sharex=True, sharey=True, squeeze=False,
    )
    axes = axes[:, 0]
    for ax, (name, data) in zip(axes, scenarios.items()):
        window = data.loc[
            (data["Minute"] >= time_start) & (data["Minute"] < time_finish)
        ]
        if window.empty:
            raise ValueError(f"No data in the requested interval for {name}.")
        # Extend the final step through the last minute of the interval.
        x = np.r_[window["Minute"].to_numpy(), window["Minute"].iloc[-1] + 1]
        series = lambda col: np.r_[window[col].to_numpy(), window[col].iloc[-1]]
        tf = series("Available TF [kW]")
        total = series("Available total [kW]")
        ax.fill_between(x, 0, tf, step="post", alpha=.65,
                        color="#4F81BD", label="Available time flexibility")
        ax.fill_between(x, tf, total, step="post", alpha=.8,
                        color="#70AD47", label="Available power flexibility")
        ax.step(x, series("Required reduction [kW]"), where="post",
                color="black", linestyle="--", label="Reduction needed to meet limit")
        failed = window.loc[window["Violation"]]
        ax.hlines(
            y=failed["Required reduction [kW]"],
            xmin=failed["Minute"],
            xmax=failed["Minute"] + 1,
            color="#C00000",
            linewidth=4,
            zorder=4,
            label="Violation after control",
        )
        ax.set_title(name)
        ax.set_ylabel("Power reduction [kW]")
        ax.grid(axis="y", alpha=.25)
        ax.set_axisbelow(True)
        ax.set_ylim(bottom=0)
    ticks = np.arange(time_start, time_finish + 1, tick_step)
    axes[-1].set_xticks(ticks)
    axes[-1].set_xticklabels([f"{int(t)//60:02d}:{int(t)%60:02d}" for t in ticks])
    axes[-1].set_xlim(time_start, time_finish)
    axes[-1].set_xlabel("Time")
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="lower center", ncol=2, frameon=False)
    fig.tight_layout(rect=(0, .10, 1, 1))
    return fig, axes


In [ ]:
def compute_utility_metrics(
    X_forecast,
    X_opt,
    Type,
    dev_names=None,
    dt_minutes=1,
):
    """Compare forecast and optimized states for power-flexible devices."""
    import numpy as np
    import pandas as pd

    X_forecast = np.asarray(X_forecast, dtype=float)
    X_opt = np.asarray(X_opt, dtype=float)

    if X_forecast.ndim != 2 or X_forecast.shape != X_opt.shape:
        raise ValueError(
            "X_forecast and X_opt must be 2D matrices with the same shape."
        )
    if X_forecast.size == 0:
        raise ValueError("X_forecast and X_opt cannot be empty.")
    if dt_minutes <= 0:
        raise ValueError("dt_minutes must be positive.")
    if not (np.all(np.isfinite(X_forecast)) and np.all(np.isfinite(X_opt))):
        raise ValueError("X_forecast and X_opt cannot contain NaN or infinity.")

    n_minutes, n_devices = X_forecast.shape
    Type = np.asarray(Type, dtype=int)
    if Type.shape != (n_devices,):
        raise ValueError("Type must contain one value per device.")
    if dev_names is None:
        names = np.array([f"Device {j}" for j in range(n_devices)])
    else:
        names = np.asarray(dev_names, dtype=object)
        if names.shape != (n_devices,):
            raise ValueError("dev_names must contain one name per device.")

    # Utility agreement is evaluated only for time-flexible devices.
    time_flexible = Type == 2
    if not time_flexible.any():
        raise ValueError("No time-flexible devices (Type == 2) were found.")

    X_forecast = X_forecast[:, time_flexible]
    X_opt = X_opt[:, time_flexible]
    names = names[time_flexible]

    equal = np.isclose(X_forecast, X_opt, rtol=0.0, atol=1e-9)
    forecast_on_opt_off = (X_forecast > 0.5) & (X_opt <= 0.5)
    forecast_off_opt_on = (X_forecast <= 0.5) & (X_opt > 0.5)

    rows = []
    for j, name in enumerate(names):
        equal_minutes = float(equal[:, j].sum() * dt_minutes)
        different_minutes = float((~equal[:, j]).sum() * dt_minutes)
        rows.append({
            "Device": str(name),
            "Compared time [min]": float(n_minutes * dt_minutes),
            "Equal time [min]": equal_minutes,
            "Equality [%]": float(100.0 * equal[:, j].mean()),
            "Different time [min]": different_minutes,
            "Forecast ON -> optimized OFF [min]": float(
                forecast_on_opt_off[:, j].sum() * dt_minutes
            ),
            "Forecast OFF -> optimized ON [min]": float(
                forecast_off_opt_on[:, j].sum() * dt_minutes
            ),
        })

    rows.append({
        "Device": "Overall",
        "Compared time [min]": float(equal.size * dt_minutes),
        "Equal time [min]": float(equal.sum() * dt_minutes),
        "Equality [%]": float(100.0 * equal.mean()),
        "Different time [min]": float((~equal).sum() * dt_minutes),
        "Forecast ON -> optimized OFF [min]": float(
            forecast_on_opt_off.sum() * dt_minutes
        ),
        "Forecast OFF -> optimized ON [min]": float(
            forecast_off_opt_on.sum() * dt_minutes
        ),
    })

    return pd.DataFrame(rows)


In [ ]:
def plot_power_share_by_flexibility_at_violations(
    scenarios,
    Type,
    Power_max,
    time_start,
    time_finish,
    tolerance_W=1e-6,
    figsize=(9, 6),
):
    """
    Plot the cumulative load composition by flexibility class
    during power-limit violation periods.

    Type:
        0 = Non-flexible
        1 = Power-flexible
        2 = Time-flexible

    For power-flexible devices:
        P = X * Y * Power_max

    For non-flexible and time-flexible devices:
        P = X * Power_max

    Percentages sum to 100% within each scenario.
    All percentage labels are displayed outside the bars.
    """

    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt

    Type = np.asarray(
        Type,
        dtype=int,
    )

    Power_max = np.asarray(
        Power_max,
        dtype=float,
    )

    if Type.shape != Power_max.shape:
        raise ValueError(
            "Type and Power_max must have the same length."
        )

    if not 0 <= time_start < time_finish:
        raise ValueError(
            "Invalid time interval."
        )

    type_labels = {
        0: "Non-flexible",
        1: "Power-flexible",
        2: "Time-flexible",
    }

    type_order = [
        "Non-flexible",
        "Power-flexible",
        "Time-flexible",
    ]

    colors = {
        "Non-flexible": "#8C8C8C",
        "Power-flexible": "#70AD47",
        "Time-flexible": "#4F81BD",
    }

    rows = []

    # ========================================================
    # ANALYSE EACH SCENARIO
    # ========================================================

    for scenario, values in scenarios.items():

        X = np.asarray(
            values["X"],
            dtype=float,
        )

        Y = np.asarray(
            values["Y"],
            dtype=float,
        )

        P_lim = np.asarray(
            values["P_lim"],
            dtype=float,
        ).reshape(-1)

        if (
            X.ndim != 2
            or X.shape != Y.shape
            or X.shape[1] != len(Power_max)
        ):
            raise ValueError(
                f"{scenario}: incompatible X/Y dimensions."
            )

        if (
            len(P_lim) != X.shape[0]
            or time_finish > X.shape[0]
        ):
            raise ValueError(
                f"{scenario}: incompatible power-limit vector."
            )

        # ====================================================
        # DEVICE POWER
        # ====================================================

        device_power = np.zeros_like(
            X,
            dtype=float,
        )

        for j in range(
            X.shape[1]
        ):

            if Type[j] == 1:

                # Power-flexible device
                device_power[:, j] = (
                    X[:, j]
                    * Y[:, j]
                    * Power_max[j]
                )

            else:

                # Non-flexible or time-flexible device
                device_power[:, j] = (
                    X[:, j]
                    * Power_max[j]
                )

        # ====================================================
        # VIOLATIONS
        # ====================================================

        total_power = (
            device_power.sum(
                axis=1
            )
        )

        violation_mask = (
            total_power[
                time_start:time_finish
            ]
            >
            P_lim[
                time_start:time_finish
            ]
            + tolerance_W
        )

        violation_count = int(
            violation_mask.sum()
        )

        power_violation = (
            device_power[
                time_start:time_finish
            ][violation_mask]
        )

        total_violation_load = (
            power_violation.sum()
        )

        # ====================================================
        # FLEXIBILITY CLASS CONTRIBUTIONS
        # ====================================================

        for device_type in [
            0,
            1,
            2,
        ]:

            type_mask = (
                Type == device_type
            )

            class_load = (
                power_violation[
                    :, type_mask
                ].sum()
            )

            share = (
                100.0
                * class_load
                / total_violation_load
                if total_violation_load > 0
                else 0.0
            )

            rows.append({
                "Scenario":
                    scenario,

                "Type":
                    type_labels[
                        device_type
                    ],

                "Cumulative load [kW min]":
                    class_load / 1000,

                "Share [%]":
                    share,

                "Violation minutes":
                    violation_count,
            })

    # ========================================================
    # SUMMARY TABLE
    # ========================================================

    summary = pd.DataFrame(
        rows
    )

    # ========================================================
    # PREPARE DATA FOR PLOT
    # ========================================================

    pivot = summary.pivot(
        index="Scenario",
        columns="Type",
        values="Share [%]",
    )

    pivot = pivot.reindex(
        index=list(
            scenarios.keys()
        ),
        columns=type_order,
        fill_value=0,
    )

    # ========================================================
    # PLOT
    # ========================================================

    fig, ax = plt.subplots(
        figsize=figsize
    )

    bottom = np.zeros(
        len(pivot)
    )

    for type_name in type_order:

        values = (
            pivot[
                type_name
            ].to_numpy()
        )

        bars = ax.bar(
            pivot.index,
            values,
            bottom=bottom,
            width=0.62,
            color=colors[
                type_name
            ],
            label=type_name,
            alpha=0.9,
        )

        # ====================================================
        # PERCENTAGE LABELS
        # All percentages outside the bars
        # ====================================================

        for bar, value, base in zip(
            bars,
            values,
            bottom,
        ):

            if value > 0:

                y_center = (
                    base
                    + value / 2
                )

                ax.text(
                    bar.get_x()
                    + bar.get_width()
                    + 0.04,
                    y_center,
                    f"{value:.1f}%",
                    ha="left",
                    va="center",
                    fontsize=9.5,
                    color="black",
                )

        bottom += values

    # ========================================================
    # VIOLATION MINUTES ABOVE BARS
    # ========================================================

    violation_counts = (
        summary
        .groupby(
            "Scenario",
            sort=False,
        )["Violation minutes"]
        .first()
    )

    for i, scenario in enumerate(
        pivot.index
    ):

        ax.text(
            i,
            101.5,
            f"{violation_counts[scenario]} min",
            ha="center",
            va="bottom",
            fontsize=10,
        )

    # ========================================================
    # FORMATTING
    # ========================================================

    ax.set_title(
        "Load composition during power-limit violations",
        fontsize=14,
        pad=12,
    )

    ax.set_ylabel(
        "Share of cumulative load during violations [%]",
        fontsize=11,
    )

    ax.set_ylim(
        0,
        107,
    )

    # Extra horizontal space for percentage labels
    ax.margins(
        x=0.18
    )

    ax.tick_params(
        axis="both",
        labelsize=10,
    )

    ax.grid(
        axis="y",
        alpha=0.3,
    )

    ax.set_axisbelow(
        True
    )

    # Remove upper and right borders
    ax.spines[
        "top"
    ].set_visible(
        False
    )

    ax.spines[
        "right"
    ].set_visible(
        False
    )

    # ========================================================
    # LEGEND
    # ========================================================

    ax.legend(
        loc="upper center",
        bbox_to_anchor=(
            0.5,
            -0.12,
        ),
        ncol=3,
        frameon=False,
        fontsize=10,
    )

    plt.tight_layout()

    plt.show()

    return (
        summary,
        fig,
        ax,
    )

In [ ]:
def compute_energy_metrics(
    P_baseline,
    P_utility,
    P_energy,
    P_lim=None,
    time_zones=None,
    tou_price_eur_kwh=None,
    dt_minutes=1,
    tolerance=1e-6,
):
    import numpy as np
    import pandas as pd

    cases = {
        "Baseline": np.asarray(P_baseline, dtype=float).flatten(),
        "Max Utility": np.asarray(P_utility, dtype=float).flatten(),
        "Min Energy": np.asarray(P_energy, dtype=float).flatten(),
    }

    n = len(cases["Baseline"])

    if n == 0 or any(len(P) != n for P in cases.values()):
        raise ValueError("Os três perfis devem ter o mesmo comprimento.")

    if dt_minutes <= 0:
        raise ValueError("dt_minutes deve ser positivo.")

    if any(not np.all(np.isfinite(P)) for P in cases.values()):
        raise ValueError("Os perfis não podem conter NaN ou infinito.")

    if P_lim is not None:
        P_lim = np.asarray(P_lim, dtype=float).flatten()

        if len(P_lim) != n or not np.all(np.isfinite(P_lim)):
            raise ValueError("P_lim deve ter o mesmo comprimento e valores válidos.")

    if (time_zones is None) != (tou_price_eur_kwh is None):
        raise ValueError(
            "time_zones and tou_price_eur_kwh must be provided together."
        )

    price_vector = None
    if time_zones is not None:
        price_vector = np.full(n, np.nan, dtype=float)
        coverage = np.zeros(n, dtype=int)

        for timezone, limits in time_zones.items():
            if timezone not in tou_price_eur_kwh:
                raise ValueError(f"Missing electricity price for timezone {timezone}.")

            start = int(limits["start"])
            end = int(limits["end"])
            price = float(tou_price_eur_kwh[timezone])

            if not (0 <= start < end <= n) or not np.isfinite(price):
                raise ValueError(f"Invalid tariff definition for timezone {timezone}.")

            price_vector[start:end] = price
            coverage[start:end] += 1

        if not np.all(coverage == 1):
            raise ValueError(
                "The tariff timezones must cover every sample exactly once."
            )

    # Converter soma de potência [W] em energia [kWh]
    factor = dt_minutes / (60 * 1000)
    E_baseline = cases["Baseline"].sum() * factor
    C_baseline = (
        float(np.sum(cases["Baseline"] * price_vector) * factor)
        if price_vector is not None else None
    )

    rows = []

    for name, P in cases.items():
        E = P.sum() * factor
        delta_E = E - E_baseline

        row = {
            "Scenario": name,
            "Total energy [kWh]": float(E),
            "ΔE vs Baseline [kWh]": float(delta_E),
            "ΔE vs Baseline [%]":
                float(100 * delta_E / E_baseline)
                if abs(E_baseline) > 1e-12 else np.nan,
        }

        if price_vector is not None:
            cost = float(np.sum(P * price_vector) * factor)
            delta_cost = cost - C_baseline
            row["Total energy cost [EUR]"] = cost
            row["ΔCost vs Baseline [EUR]"] = delta_cost
            row["ΔCost vs Baseline [%]"] = (
                float(100 * delta_cost / C_baseline)
                if abs(C_baseline) > 1e-12 else np.nan
            )

        if P_lim is not None:
            violations = P > P_lim + tolerance
            excess = np.where(violations, P - P_lim, 0.0)

            row["Excess energy above limit [kWh]"] = float(
                excess.sum() * factor
            )
            row["Energy during violation periods [kWh]"] = float(
                P[violations].sum() * factor
            )

        rows.append(row)

    return pd.DataFrame(rows)

In [ ]:
def plot_time_flexible_comparison(
    device,
    dev_names,
    Type,
    W_baseline,
    X_baseline,
    W_utility,
    X_utility,
    W_energy,
    X_energy,
    time_start="00:00",
    time_finish="24:00",
    tick_step=15,
    figsize=(14, 4),
):
    import numpy as np
    import matplotlib.pyplot as plt

    def to_minute(value):
        if isinstance(value, str):
            hour, minute = map(int, value.split(":"))
            if not (
                0 <= hour <= 24
                and 0 <= minute < 60
                and (hour < 24 or minute == 0)
            ):
                raise ValueError(f"Hora inválida: {value}")
            return hour * 60 + minute
        return int(value)

    # Selecionar equipamento por nome ou índice
    if isinstance(device, str):
        normalize = lambda name: "".join(str(name).lower().split())
        names = [normalize(name) for name in dev_names]

        if normalize(device) not in names:
            raise ValueError(f"Equipamento desconhecido: {device}")

        j = names.index(normalize(device))
    else:
        j = int(device)

    if not 0 <= j < len(dev_names):
        raise ValueError("Índice de equipamento inválido.")

    if Type[j] != 2:
        raise ValueError(f"{dev_names[j]} não é time-flexible.")

    start = to_minute(time_start)
    end = to_minute(time_finish)
    tick_step = int(tick_step)

    if tick_step <= 0:
        raise ValueError("tick_step deve ser positivo.")

    # Desenhar o laranja primeiro
    scenarios = [
        ("Max Utility", W_utility, "#F79646", "-", 3.5),
        ("Baseline", W_baseline, "#4F81BD", "--", 2),
        ("Min Energy", W_energy, "#548235", "dashdot", 2.5),
    ]

    for label, W, _, _, _ in scenarios:
        W = np.asarray(W)

        if W.ndim != 2 or W.shape[1] != len(dev_names):
            raise ValueError(f"{label}: dimensão de W inválida.")

        if not 0 <= start < end <= len(W):
            raise ValueError(f"{label}: intervalo temporal inválido.")

    fig, ax = plt.subplots(figsize=figsize)

    for label, W, color, style, width in scenarios:
        W = np.asarray(W)
        # Incluir o ponto final quando disponível
        stop = min(end + 1, len(W))

        ax.plot(
            np.arange(start, stop),
            W[start:stop, j],
            label=label,
            color=color,
            linestyle=style,
            linewidth=width,
        )

    ax.axhline(1, color="gray", linestyle="--", linewidth=1)
    ax.axhline(10, color="gray", linestyle="--", linewidth=1)

    ticks = np.r_[np.arange(start, end, tick_step), end]
    ax.set_xticks(ticks)
    ax.set_xticklabels(
        [f"{t // 60:02d}:{t % 60:02d}" for t in ticks],
        rotation=45,
        ha="right",
    )

    ax.set_xlim(start, end)
    ax.set_xlabel("Time")
    ax.set_ylabel(r"Time weight $W_T$")
    ax.set_title(f"Time weight — {dev_names[j]}")
    ax.grid(alpha=0.25)

    handles, labels = ax.get_legend_handles_labels()
    order = [1, 0, 2]
    ax.legend(
        [handles[i] for i in order],
        [labels[i] for i in order],
        ncol=3,
    )

    fig.tight_layout()
    plt.show()

    return fig, ax

In [ ]:
def plot_power_flexible_Y_comparison(
    device,
    dev_names,
    Type,
    X_baseline,
    Y_baseline,
    X_utility,
    Y_utility,
    X_energy,
    Y_energy,
    time_start=0,
    time_finish=None,
    tick_step=15,
):
    import numpy as np
    import matplotlib.pyplot as plt

    # Selecionar equipamento por nome ou índice
    if isinstance(device, str):
        normalize = lambda name: "".join(str(name).lower().split())
        names = [normalize(name) for name in dev_names]
        j = names.index(normalize(device))
    else:
        j = int(device)

    if Type[j] != 1:
        raise ValueError(f"{dev_names[j]} não é power-flexible.")

    start = int(time_start)
    end = len(X_baseline) if time_finish is None else int(time_finish)
    tick_step = int(tick_step)

    scenarios = [
        ("Max Utility", X_utility, Y_utility, "#F79646", "-", 3.5),
        ("Baseline", X_baseline, Y_baseline, "#4F81BD", "--", 2),
        ("Min Energy", X_energy, Y_energy, "#548235", "dashdot", 2.5),
    ]

    if tick_step <= 0:
        raise ValueError("tick_step deve ser positivo.")

    for _, X, Y, _, _, _ in scenarios:        
        if np.shape(X) != np.shape(Y):
            raise ValueError("X e Y devem ter as mesmas dimensões.")
        if not 0 <= start < end <= len(X):
            raise ValueError("Intervalo temporal inválido.")

    fig, ax = plt.subplots(figsize=(14, 4))
    time = np.arange(start, end + 1)

    for label, X, Y, color, linestyle, linewidth in scenarios:
        X = np.asarray(X)
        Y = np.asarray(Y)

        values = 100 * np.where(
            X[start:end, j] == 1,
            Y[start:end, j],
            0.0,
        )

        ax.step(
            time,
            np.r_[values, values[-1]],
            where="post",
            label=label,
            color=color,
            linestyle=linestyle,
            linewidth=linewidth,
        )

    ticks = np.r_[np.arange(start, end, tick_step), end]

    ax.set_xticks(ticks)
    ax.set_xticklabels(
        [f"{t // 60:02d}:{t % 60:02d}" for t in ticks],
        rotation=45,
        ha="right",
    )

    ax.set_xlim(start, end)
    ax.set_ylim(-5, 105)
    ax.set_xlabel("Time")
    ax.set_ylabel("Relative power $Y$ [%]")
    ax.set_title(f"Power modulation — {dev_names[j]}")
    ax.grid(alpha=0.25)
    ax.legend()

    fig.tight_layout()
    plt.show()

    return fig, ax

In [ ]:


def plot_fridge_temperature_smartfridge(
    T_baseline,
    T_opt,
    X_baseline,
    X_opt,
    SP=4.0,
    H_minus=2.0,
    H_plus=6.0,
    tick_step=60,
):
    n = len(T_opt)
    x = np.arange(n)
    labels = [f"{m//60:02d}:{m%60:02d}" for m in x]

    fig, ax1 = plt.subplots(figsize=(18, 6))

    ax1.plot(x, T_baseline, linewidth=3, color="#4F81BD", label="Temperature baseline")
    ax1.plot(x, T_opt, linewidth=3, color="#F79646", label="Temperature optimized")

    ax1.axhline(SP, linestyle=":", linewidth=2.5, color="black", label="Setpoint")
    ax1.axhline(H_minus, linestyle="--", linewidth=2, color="gray", label="H- lower limit")
    ax1.axhline(H_plus, linestyle="--", linewidth=2, color="gray", label="H+ upper limit")

    ax1.set_xlabel("Time", fontsize=18, fontweight="bold")
    ax1.set_ylabel("Temperature [°C]", fontsize=18, fontweight="bold")

    ax1.set_xticks(np.arange(0, n, tick_step))
    ax1.set_xticklabels(labels[::tick_step], rotation=90, fontsize=11)

    ax1.grid(axis="y", alpha=0.3)

    ax2 = ax1.twinx()

    ax2.step(x, X_baseline, where="post", linewidth=3, color="#4F81BD", alpha=0.30, label="X baseline")
    ax2.step(x, X_opt, where="post", linewidth=3, color="#F79646", alpha=0.30, label="X optimized")

    ax2.set_ylabel("Compressor Status", fontsize=18, fontweight="bold")
    ax2.set_ylim(-0.1, 1.1)
    ax2.set_yticks([0, 1])

    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()

    # Arrange the legend in three visual groups. The two invisible
    # entries keep the setpoint centred between the thermal limits.
    from matplotlib.lines import Line2D
    blank = Line2D([], [], linestyle="none", label="")
    legend_lines = [
        lines1[0], lines1[1], blank,
        lines1[3], lines1[2], lines1[4],
        lines2[0], lines2[1], blank,
    ]
    legend_labels = [
        labels1[0], labels1[1], "",
        labels1[3], labels1[2], labels1[4],
        labels2[0], labels2[1], "",
    ]

    ax1.legend(
        legend_lines,
        legend_labels,
        loc="upper center",
        bbox_to_anchor=(0.5, -0.18),
        ncol=3,
        frameon=False,
        fontsize=15
    )

    plt.tight_layout()


    plt.show()

In [ ]:
def compute_fridge_temperature_metrics(
    T_baseline,
    T_utility,
    T_energy,
    SP=4.0,
    H_minus=2.0,
    H_plus=6.0,
    dt_minutes=1
):
    import numpy as np
    import pandas as pd

    cases = {
        "Baseline": np.asarray(T_baseline, dtype=float),
        "RT - Max Utility": np.asarray(T_utility, dtype=float),
        "RT - Min Energy": np.asarray(T_energy, dtype=float),
    }

    rows = []

    for name, T in cases.items():

        above_upper = T > H_plus
        below_lower = T < H_minus

        rows.append({
            "Case": name,

            "Mean temperature [°C]":
                float(np.mean(T)),

            "Max temperature [°C]":
                float(np.max(T)),

            "Min temperature [°C]":
                float(np.min(T)),

            "Mean abs. deviation from SP [°C]":
                float(np.mean(np.abs(T - SP))),
        })

    return pd.DataFrame(rows)

In [ ]:
def plot_stacked_device_power(
    X_matrix,
    Y_matrix,
    Power_max,
    dev_names,
    time_start=0,
    time_finish=None,
    P_lim_vector=None,
    colors=None,
    tick_step=15,
    title="Stacked Device Power"
):
    """
    Plot stacked device-level power for a selected time interval.

    Parameters
    ----------
    X_matrix : ndarray, shape (T, N_DEV)
        ON/OFF state matrix.

    Y_matrix : ndarray, shape (T, N_DEV)
        Power modulation matrix.

    Power_max : array-like, shape (N_DEV,)
        Rated/max power of each device [W].

    dev_names : list of str
        Device names.

    time_start : int
        Start minute of the interval.

    time_finish : int or None
        End minute of the interval.
        If None, uses the full remaining horizon.

    P_lim_vector : array-like, optional
        Minute-level power limit [W].

    colors : list, optional
        Device colors.

    tick_step : int
        X-axis tick spacing in minutes.

    title : str
        Plot title.

    Returns
    -------
    P_dev : ndarray
        Device-level power matrix for the full horizon.

    P_total : ndarray
        Aggregate power vector for the full horizon.
    """

    import numpy as np
    import matplotlib.pyplot as plt

    X_matrix = np.asarray(X_matrix, dtype=float)
    Y_matrix = np.asarray(Y_matrix, dtype=float)
    Power_max = np.asarray(Power_max, dtype=float)

    T, N_DEV = X_matrix.shape

    if time_finish is None:
        time_finish = T

    if not (0 <= time_start < time_finish <= T):
        raise ValueError(
            f"Invalid interval: "
            f"time_start={time_start}, "
            f"time_finish={time_finish}, "
            f"T={T}"
        )

    if Y_matrix.shape != X_matrix.shape:
        raise ValueError(
            "X_matrix and Y_matrix must have the same shape."
        )

    if len(Power_max) != N_DEV:
        raise ValueError(
            "Power_max length must match number of devices."
        )

    if len(dev_names) != N_DEV:
        raise ValueError(
            "dev_names length must match number of devices."
        )

    # ========================================================
    # DEVICE POWER
    # ========================================================

    P_dev = (
        X_matrix
        * Y_matrix
        * Power_max.reshape(1, -1)
    )

    P_total = np.sum(
        P_dev,
        axis=1
    )

    # ========================================================
    # SELECT INTERVAL
    # ========================================================

    P_dev_slice = P_dev[
        time_start:time_finish,
        :
    ]

    minutes = np.arange(
        time_start,
        time_finish
    )

    x = np.arange(
        len(minutes)
    )

    # ========================================================
    # KEEP ONLY DEVICES ACTIVE IN THIS INTERVAL
    # ========================================================

    active_mask = np.any(
        P_dev_slice > 1e-9,
        axis=0
    )

    active_indices = np.where(
        active_mask
    )[0]

    P_dev_active = P_dev_slice[
        :,
        active_indices
    ]

    active_names = [
        dev_names[j]
        for j in active_indices
    ]

    # ========================================================
    # COLORS
    # ========================================================

    active_colors = None

    if colors is not None:

        active_colors = [
            colors[j]
            for j in active_indices
        ]

    # ========================================================
    # PLOT
    # ========================================================

    fig, ax = plt.subplots(
        figsize=(16, 7)
    )

    if len(active_indices) > 0:

        ax.stackplot(
            x,
            P_dev_active.T,
            labels=active_names,
            colors=active_colors,
            alpha=0.9
        )

    # Power limit
    if P_lim_vector is not None:

        P_lim_vector = np.asarray(
            P_lim_vector,
            dtype=float
        )

        P_lim_slice = P_lim_vector[
            time_start:time_finish
        ]

        ax.plot(
            x,
            P_lim_slice,
            linestyle="--",
            linewidth=2,
            label="P_lim"
        )

    # ========================================================
    # TIME LABELS
    # ========================================================

    tick_positions = np.arange(
        0,
        len(minutes),
        tick_step
    )

    tick_minutes = minutes[
        tick_positions
    ]

    tick_labels = [
        f"{m // 60:02d}:{m % 60:02d}"
        for m in tick_minutes
    ]

    ax.set_xticks(
        tick_positions
    )

    ax.set_xticklabels(
        tick_labels,
        rotation=45,
        ha="right"
    )

    # ========================================================
    # FORMATTING
    # ========================================================

    start_label = (
        f"{time_start // 60:02d}:"
        f"{time_start % 60:02d}"
    )

    finish_label = (
        f"{time_finish // 60:02d}:"
        f"{time_finish % 60:02d}"
    )

    ax.set_title(
        f"{title} | "
        f"{start_label} - {finish_label}"
    )

    ax.set_xlabel(
        "Time"
    )

    ax.set_ylabel(
        "Power [W]"
    )

    ax.grid(
        axis="y",
        alpha=0.3
    )

    ax.legend(
        loc="center left",
        bbox_to_anchor=(1.02, 0.5),
        frameon=False
    )

    plt.tight_layout()
    plt.show()

    return P_dev, P_total

In [ ]:
def compute_power_limit_metrics(
    P_baseline,
    P_utility,
    P_energy,
    P_lim,
    dt_minutes=1,
    tolerance=1e-6,
):
    import numpy as np
    import pandas as pd

    P_lim = np.asarray(P_lim, dtype=float).flatten()

    cases = {
        "Baseline": P_baseline,
        "Max Utility": P_utility,
        "Min Energy": P_energy,
    }

    rows = []

    for name, P in cases.items():
        P = np.asarray(P, dtype=float).flatten()

        if len(P) == 0 or len(P) != len(P_lim):
            raise ValueError("P e P_lim devem ter o mesmo comprimento.")

        excess = np.maximum(P - P_lim, 0.0)
        violations = excess > tolerance
        excess = np.where(violations, excess, 0.0)

        rows.append({
            "Scenario": name,
            "Violation time [min]":
                float(violations.sum() * dt_minutes),
            "Violation time [%]":
                float(100 * violations.mean()),
            "Max exceedance [W]":
                float(excess.max()),
            "Mean exceedance [W]":
                float(excess[violations].mean())
                if violations.any() else 0.0,
            "Excess energy [Wh]":
                float(excess.sum() * dt_minutes / 60),
            "Peak power [W]":
                float(P.max()),
            "Mean power [W]":
                float(P.mean()),
        })

    return pd.DataFrame(rows)

In [ ]:
def get_p_lim_for_minute(
    minute,
    pv_bess=False
):
    """
    Instantaneous power limit [W] for a given minute.
    """

    tz = get_time_zone(
        minute
    )

    return (
        timezone_power_limits_W[
            tz
        ])
    
def get_time_zone(minute):
    """
    Return the time-zone index associated with a minute of the day.

    0 -> 00:00 - 09:59
    1 -> 10:00 - 19:29
    2 -> 19:30 - 22:29
    3 -> 22:30 - 23:59
    """

    minute = int(minute)

    for tz, limits in TIME_ZONES.items():

        if (
            limits["start"]
            <= minute
            < limits["end"]
        ):
            return tz

    raise ValueError(
        f"Minute outside daily horizon: {minute}"
    )


# Data preprocessing

## Battery Info

In [ ]:
def characterize_battery(
    blocks,
    capacity_kWh,
    soc_init_kWh,
    soc_min_kWh,
    soc_max_kWh,
    charge_power_max_kW,
    discharge_power_max_kW,
    eta_charge,
    eta_discharge,
    block_size_min=15
):
    """
    Creates the physical characterization of a battery for each
    optimization block.

    Parameters
    ----------
    blocks : iterable
        Optimization block identifiers.

    capacity_kWh : float
        Nominal or usable battery capacity [kWh].

    soc_init_kWh : float
        Initial stored energy at the beginning of the horizon [kWh].

    soc_min_kWh : float
        Minimum allowed stored energy [kWh].

    soc_max_kWh : float
        Maximum allowed stored energy [kWh].

    charge_power_max_kW : float
        Maximum battery charging power [kW].

    discharge_power_max_kW : float
        Maximum battery discharging power [kW].

    eta_charge : float
        Charging efficiency, between 0 and 1.

    eta_discharge : float
        Discharging efficiency, between 0 and 1.

    block_size_min : int or float
        Duration of each optimization block [minutes].

    Returns
    -------
    battery_characterization : pandas.DataFrame
        Battery physical parameters for each optimization block.

    battery_info : dict
        Global battery parameters.
    """

    # -------------------------
    # Input validation
    # -------------------------
    if capacity_kWh <= 0:
        raise ValueError("capacity_kWh must be greater than zero.")

    if not 0 <= soc_min_kWh <= soc_init_kWh <= soc_max_kWh:
        raise ValueError(
            "The SoC values must satisfy: "
            "soc_min_kWh <= soc_init_kWh <= soc_max_kWh."
        )

    if soc_max_kWh > capacity_kWh:
        raise ValueError(
            "soc_max_kWh cannot be greater than capacity_kWh."
        )

    if charge_power_max_kW < 0:
        raise ValueError(
            "charge_power_max_kW cannot be negative."
        )

    if discharge_power_max_kW < 0:
        raise ValueError(
            "discharge_power_max_kW cannot be negative."
        )

    if not 0 < eta_charge <= 1:
        raise ValueError(
            "eta_charge must be between 0 and 1."
        )

    if not 0 < eta_discharge <= 1:
        raise ValueError(
            "eta_discharge must be between 0 and 1."
        )

    if block_size_min <= 0:
        raise ValueError(
            "block_size_min must be greater than zero."
        )

    # -------------------------
    # Time conversion
    # -------------------------
    block_duration_h = block_size_min / 60.0

    # Maximum grid-side energy exchanged during one block
    charge_energy_max_kWh = (
        charge_power_max_kW
        * block_duration_h
    )

    discharge_energy_max_kWh = (
        discharge_power_max_kW
        * block_duration_h
    )

    # -------------------------
    # SoC percentages
    # -------------------------
    soc_init_pct = (
        100
        * soc_init_kWh
        / capacity_kWh
    )

    soc_min_pct = (
        100
        * soc_min_kWh
        / capacity_kWh
    )

    soc_max_pct = (
        100
        * soc_max_kWh
        / capacity_kWh
    )

    # -------------------------
    # Per-block characterization
    # -------------------------
    battery_characterization = pd.DataFrame({
        "block": list(blocks),
        "block_size_min": block_size_min,
        "block_duration_h": block_duration_h,
        "capacity_kWh": capacity_kWh,
        "soc_min_kWh": soc_min_kWh,
        "soc_max_kWh": soc_max_kWh,
        "charge_power_max_kW": charge_power_max_kW,
        "discharge_power_max_kW": discharge_power_max_kW,
        "charge_energy_max_kWh": charge_energy_max_kWh,
        "discharge_energy_max_kWh": discharge_energy_max_kWh,
        "eta_charge": eta_charge,
        "eta_discharge": eta_discharge
    })

    # -------------------------
    # Global information
    # -------------------------
    battery_info = {
        "capacity_kWh": capacity_kWh,
        "soc_init_kWh": soc_init_kWh,
        "soc_min_kWh": soc_min_kWh,
        "soc_max_kWh": soc_max_kWh,
        "soc_init_pct": soc_init_pct,
        "soc_min_pct": soc_min_pct,
        "soc_max_pct": soc_max_pct,
        "charge_power_max_kW": charge_power_max_kW,
        "discharge_power_max_kW": discharge_power_max_kW,
        "charge_energy_max_kWh": charge_energy_max_kWh,
        "discharge_energy_max_kWh": discharge_energy_max_kWh,
        "eta_charge": eta_charge,
        "eta_discharge": eta_discharge,
        "block_size_min": block_size_min,
        "block_duration_h": block_duration_h
    }

    return battery_characterization, battery_info

battery_characterization, battery_info = characterize_battery(
    blocks=range(N_BLOCKS),
    capacity_kWh=20.0,
    soc_init_kWh=10.0,
    soc_min_kWh=2.0,
    soc_max_kWh=18.0,
    charge_power_max_kW=5.0,
    discharge_power_max_kW=5.0,
    eta_charge=0.95,
    eta_discharge=0.95,
    block_size_min=BLOCK_SIZE
)

battery_characterization.head()

## Consumption Data

In [ ]:
Year=2019
Month=2
Day=14
N_DEV = 13

# instante inicial e final do dia em análise
day_start_dt = datetime(Year, Month, Day, 0, 0, 0)
day_end_dt = day_start_dt + timedelta(days=1)

# =========================
# DEVICE GROUPS
# =========================
NON_OPERATIONAL_DEVICES = {0, 1, 2, 3, 4}  # Drinks Refrigerator, Refrigerators 2–4, Garde Manger
OPERATIONAL_DEVICES = {5, 6}  # Dual Fryer, Salamander
MISSING_STATE_DURATION = 1e-6  # minutos; não arredondar para um minuto
NON_FLEXIBLE_DEVICES = {7, 8, 9}  # Glass Washer, Dish Washer, Convection Oven
POWER_FLEXIBLE_DEVICES = {10, 11, 12}  # Light1, Light2, Exhaust
EXTERNAL_PROFILE_DEVICES = {12}  # Exhaust

POWER_FLEXIBLE_SCHEDULES = {
    10: {
        "name": "Light1",
        "start": 10 * 60,          # 10:00
        "end": 22 * 60 + 30        # 22:30
    },
    11: {
        "name": "Light2",
        "start": 10 * 60,          # 10:00
        "end": 22 * 60 + 30        # 22:30
    }
}
EXTERNAL_PROFILE_DATES = {
    12: "2019-03-11"
}

N_MIN_ACTIVATIONS = 2
T_NO_ACTIVITY_FALLBACK = 5000
T_MIN = 5
T_OFF_FORCED_ON = 5
DAY_MINUTES = 24 * 60
T= DAY_MINUTES

# diretório dos ficheiros
diretorio = '../Data'

# mapeamento appliance -> ficheiro de ativações
activation_files = {
    0: "activations_Drinks_Refrigerator.csv",
    1: "activations_Refrigerator_2.csv",
    2: "activations_Refrigerator_3.csv",
    3: "activations_Refrigerator_4.csv",
    4: "activations_Garde_Manger_10.csv",
    5: "activations_Dual_Fryer.csv",
    6: "activations_Salamander_1.csv",
    7: "activations_Glass_Washer.csv",
    8: "activations_Dish_Washer.csv",
    9: "activations_Convection_Oven_1.csv",
}
external_activation_files = {
    12: "05_Exhaust_activations.xlsx"
}

DEVICE_NAMES = {
    0: "Drinks Refrigerator",
    1: "Refrigerator 2",
    2: "Refrigerator 3",
    3: "Refrigerator 4",
    4: "Garde Manger",
    5: "Dual Fryer",
    6: "Salamander",
    7: "Glass Washer",
    8: "Dish Washer",
    9: "Convection Oven",
    10: "Light1",
    11: "Light2",
    12: "Exhaust"
}

colors = [
    "#4F81BD",  # Device 0
    "#C0504D",  # Device 1
    "#9BBB59",  # Device 2
    "#8064A2",  # Device 3
    "#4BACC6",  # Device 4
    "#F79646",  # Device 5
    "#385D8A",  # Device 6
    "#943634",  # Device 7
    "#76923C",  # Device 8
    "#5F497A",  # Device 9
    "#FFD700",  # Device 10
    "#EEFF00",  # Device 11
    "#708090",  # Device 12
]

In [ ]:
# ======================================
# LEITURA DO INFO.XLSX
# ======================================

df_info = pd.read_excel('../Data/Info.xlsx')

print("Colunas do Info.xlsx:")
print(df_info.columns.tolist())

# se quiseres confirmar o número de equipamentos do Excel
print("\nNúmero de linhas no Info.xlsx:", len(df_info))

# opcional: garantir consistência
if len(df_info) != N_DEV:
    raise ValueError(f"N_DEV={N_DEV}, mas o Info.xlsx tem {len(df_info)} linhas.")

# parâmetros por equipamento
dev_names = df_info["Dev"].tolist()
if dev_names != [DEVICE_NAMES[j] for j in range(N_DEV)]:
    raise ValueError("A ordem dos equipamentos no Info.xlsx não coincide com DEVICE_NAMES.")

# índice do tipo de flexibilidade
Type = df_info["Type"].to_numpy(dtype=int)

# estado inicial
# para já vamos ler do Excel; mais tarde podes forçar a zero ou atualizar dinamicamente
State_Init = df_info["State_Init"].to_numpy(dtype=int)

# potência máxima
Power_max = df_info["Power_max"].to_numpy(dtype=float)

# pesos da componente de potência
Weight_P_max = df_info["Weight_P_max"].to_numpy(dtype=float)
Weight_P_min = df_info["Weight_P_min"].to_numpy(dtype=float)

# percentagens de potência
# MUITO IMPORTANTE: converter de % para fração
Y_max = df_info["Y_max (%)"].to_numpy(dtype=float) / 100.0
Y_min = df_info["Y_min (%)"].to_numpy(dtype=float) / 100.0

# tempo mínimo para manter alteração de potência
TimeHoldPower = df_info["TimeHoldPower"].to_numpy(dtype=float)

# impressão de controlo
print("\nResumo dos equipamentos:")
for j in range(N_DEV):
    print(
        f"{j}: {dev_names[j]} | "
        f"Type={Type[j]} | "
        f"State_Init={State_Init[j]} | "
        f"Pmax={Power_max[j]} | "
        f"Ymin={Y_min[j]} | "
        f"Ymax={Y_max[j]}"
        f" | TimeHoldPower={TimeHoldPower[j]}"
    )

In [ ]:
def minute_of_day(ts):
    m = ts.hour * 60 + ts.minute
    if ts.second >= 30:
        m += 1
    return min(m, 1439)

def load_activation_day_csv(diretorio, nome_do_arquivo, target_date):
    caminho_arquivo = os.path.join(diretorio, nome_do_arquivo)
    if not os.path.exists(caminho_arquivo):
        raise FileNotFoundError(f"File not found: {caminho_arquivo}")

    # Os CSVs usam separadores e capitalização diferentes.
    df = pd.read_csv(caminho_arquivo, sep=None, engine="python").iloc[:, 0:3].copy()
    df.columns = df.columns.str.strip().str.lower()

    df["first_min_on"] = pd.to_datetime(df["first_min_on"], utc=True, errors="coerce")
    df["last_min_on"]  = pd.to_datetime(df["last_min_on"],  utc=True, errors="coerce")

    df["first_min_on"] = df["first_min_on"].dt.tz_convert(None)
    df["last_min_on"]  = df["last_min_on"].dt.tz_convert(None)

    df = df.dropna(subset=["first_min_on", "last_min_on"]).sort_values("first_min_on").reset_index(drop=True)

    target_date = pd.to_datetime(target_date).date()
    df_day = df[df["first_min_on"].dt.date == target_date].copy().reset_index(drop=True)

    return df_day
def build_X_day_for_device(j, nome_do_arquivo, diretorio, target_date, X_day):
    """
    Constrói X_day[:, j] para o dia inteiro.
    """
    X_day[:, j] = 0

    df_day = load_activation_day_csv(diretorio, nome_do_arquivo, target_date)

    if df_day.empty:
        return

    for _, row in df_day.iterrows():
        start = minute_of_day(row["first_min_on"])
        end   = minute_of_day(row["last_min_on"])

        if end >= start:
            X_day[start:end + 1, j] = 1
        else:
            X_day[start:1440, j] = 1
            X_day[0:end + 1, j] = 1
def build_external_profile_from_activation_file(
    diretorio,
    nome_do_arquivo,
    target_date,
    T=1440
):

    caminho_arquivo = os.path.join(diretorio, nome_do_arquivo)

    if not os.path.exists(caminho_arquivo):
        raise FileNotFoundError(f"File not found: {caminho_arquivo}")

    df = pd.read_excel(caminho_arquivo).iloc[:, 0:3].copy()

    df.columns = df.columns.str.strip()

    df["First_min_on"] = pd.to_datetime(
        df["First_min_on"],
        utc=True,
        errors="coerce"
    )

    df["Last_min_on"] = pd.to_datetime(
        df["Last_min_on"],
        utc=True,
        errors="coerce"
    )

    df["First_min_on"] = df["First_min_on"].dt.tz_convert(None)
    df["Last_min_on"] = df["Last_min_on"].dt.tz_convert(None)

    # IMPORTANTÍSSIMO
    target_date = pd.to_datetime(target_date).date()

    df = df[
        df["First_min_on"].dt.date == target_date
    ].copy()

    x = np.zeros(T, dtype=int)

    for _, row in df.iterrows():

        start = minute_of_day(row["First_min_on"])
        end = minute_of_day(row["Last_min_on"])

        if end >= start:
            x[start:end + 1] = 1
        else:
            x[start:T] = 1
            x[0:end + 1] = 1

    return x

def build_power_flexible_schedule(T, start_minute, end_minute):
    x = np.zeros(T, dtype=int)

    if end_minute > start_minute:
        x[start_minute:end_minute] = 1
    else:
        # caso atravesse meia-noite
        x[start_minute:T] = 1
        x[0:end_minute] = 1

    return x

In [ ]:
# 1) Construir X_day global apenas para devices com ficheiro
X_day = np.zeros((T, N_DEV), dtype=int)

for j, nome_do_arquivo in activation_files.items():
    build_X_day_for_device(
        j=j,
        nome_do_arquivo=nome_do_arquivo,
        diretorio=diretorio,
        target_date=day_start_dt.date(),
        X_day=X_day
    )

# 2) Inicializar matrizes
X_onoff_FC_matrix = np.zeros((T, N_DEV), dtype=int)

target_date = day_start_dt.date()

# 3) Preencher baseline com ativações reais
X_onoff_FC_matrix[:, :] = X_day[:T, :]

print("\nX_onoff_FC_matrix (baseline) for the day:")
print(X_onoff_FC_matrix)

In [ ]:
# 1) Adicionar devices power-flexible simulados
for j in POWER_FLEXIBLE_DEVICES:

    # só os que têm horário definido manualmente
    if j not in POWER_FLEXIBLE_SCHEDULES:
        continue

    sched = POWER_FLEXIBLE_SCHEDULES[j]

    X_onoff_FC_matrix[:, j] = build_power_flexible_schedule(
        T=T,
        start_minute=sched["start"],
        end_minute=sched["end"]
    )
# 2) Adicionar perfis externos
for j in EXTERNAL_PROFILE_DEVICES:

    X_onoff_FC_matrix[:, j] = build_external_profile_from_activation_file(
        diretorio=diretorio,
        nome_do_arquivo=external_activation_files[j],
        target_date=EXTERNAL_PROFILE_DATES[j],
        T=T
    )
    
# =========================
# BUILD Y_FC_MATRIX
# =========================

Y_FC_matrix = np.zeros((T, N_DEV), dtype=float)

for j in range(N_DEV):

    if Type[j] == 1:  # power-flexible
        Y_FC_matrix[:, j] = X_onoff_FC_matrix[:, j] * Y_max[j]

    else:
        # Para devices não power-flexible, Y não é relevante.
        # Mas pode ficar igual a X para manter consistência.
        Y_FC_matrix[:, j] = X_onoff_FC_matrix[:, j]

# Device characterization

## Device characterization

- **Non-Operational**: frigoríficos e garde manger, caracterização no dia inteiro.
- **Operational**: salamandra e dual fryer, caracterização apenas entre a primeira e a última amostra ON, incluindo os períodos OFF internos.

Aplicam-se as mesmas regras de `Characterization.ipynb`: `INCOMPLETE_POLICY` seleciona `observed` ou `nearest`; numa janela apenas ON, `T_ON` é a duração da janela e `T_OFF = MISSING_STATE_DURATION` (`1e-6` minutos). Pesos e durações são `NaN` fora da janela. Sem ativações, ficam a `NaN` durante todo o dia.

A máscara `time_flex_available_matrix`, definida pela janela de referência, determina onde o RT pode atuar. Fora dela, ambos os modelos mantêm `X_opt = X_in` e ignoram os limiares do peso anterior. O `NaN` é apenas a representação de peso não aplicável: não entra no solver, onde o coeficiente correspondente é zero e o estado está fixo.

Ao abrir a janela, o peso anterior à primeira decisão é inicializado em 10 para uma referência ON (1 para OFF), evoluindo uma vez com a decisão executada. Não há clipping nem arredondamento das durações para um minuto. A inicialização dos Non-Operational mantém-se como anteriormente.

In [ ]:
@dataclass
class EpisodeT:
    """
    OFF-ON appliance episode.

    For complete episodes, both the OFF and ON blocks are observed.

    With nearest, only complete OFF -> ON episodes are returned,
    except for single-state operational windows with an epsilon duration.
    The observed/one policies also return completed boundary segments.

    start
        First observed index belonging to the episode.

    end
        Last observed index belonging to the episode.

    T_OFF
        Observed or estimated OFF duration.

    T_ON
        Observed or estimated ON duration.

    timezone
        Timezone associated with the episode.
    """

    start: int
    end: int

    T_OFF: float
    T_ON: float

    timezone: int


def get_operation_window(X):
    """Return first-to-last ON sample as a slice (stop is exclusive).

    No ON samples means an empty window. Internal OFF blocks are retained.
    """
    X = np.asarray(X)
    if X.ndim != 1 or not np.all(np.isin(X, [0, 1])):
        raise ValueError("X must be a one-dimensional binary profile.")
    active = np.flatnonzero(X == 1)
    if active.size == 0:
        return slice(0, 0)
    return slice(int(active[0]), int(active[-1]) + 1)


def extract_off_on_episodes(
    X,
    incomplete_policy="observed",
    minimum_duration=1, 
    index_offset=0,
    timezone=0,
    operational=False,
    missing_duration=1e-6
):
    """
    Extract OFF-ON episodes and complete boundary episodes according
    to an incomplete-episode policy.

    Complete episode
    ----------------
    A complete episode consists of:

        OFF block -> ON block

    Its durations are assigned directly:

        T_OFF = observed OFF-block duration
        T_ON  = observed ON-block duration

    Incomplete episodes
    -------------------
    Initial ON block:
        The preceding OFF block is not observed.

    Final OFF block:
        The following ON block is not observed.

    Policies
    --------
    observed:
        The missing duration is set equal to the duration of the
        observed block.

    nearest:
        Only complete OFF -> ON episodes are returned. Boundary samples
        inherit BOTH durations from the nearest complete episode in
        compute_ton_toff. Boundary stretches are not counted as episodes.

    one:
        The missing duration is set to minimum_duration.

    Parameters
    ----------
    X : array-like, shape (T,)
        Binary appliance-state vector.

    index_offset : int, default=0
        Offset used to convert local indices into global indices.

    incomplete_policy : {"observed", "nearest", "one"},
        default="observed"
        Policy used to complete boundary episodes.

    minimum_duration : int, default=1
        Minimum accepted duration for ordinary episodes.

    operational : bool, default=False
        Restrict episodes to the first-to-last ON window. If this window
        has only one state, return one episode with missing_duration for
        the absent state, including when the policy is nearest.

    missing_duration : float, default=1e-6
        Positive numerical substitute for a state absent from the entire
        operational window. Does not replace durations in ordinary cycles.

    Returns
    -------
    episodes : list of EpisodeT
        Complete episodes for nearest; complete episodes and completed
        boundary segments for observed/one, ordered in time.
    """

    X = np.asarray(X, dtype=int)

    if X.ndim != 1:
        raise ValueError(
            "X must be one-dimensional."
        )

    if not np.all(np.isin(X, [0, 1])):
        raise ValueError(
            "X must contain only 0 and 1."
        )

    valid_policies = {
        "observed",
        "nearest",
        "one",
    }

    if incomplete_policy not in valid_policies:
        raise ValueError(
            "incomplete_policy must be "
            "'observed', 'nearest' or 'one'."
        )

    if minimum_duration < 1:
        raise ValueError(
            "minimum_duration must be at least 1."
        )

    if not np.isfinite(missing_duration) or missing_duration <= 0:
        raise ValueError("missing_duration must be finite and positive.")

    if operational:
        window = get_operation_window(X)
        X = X[window]
        index_offset += window.start

    T = len(X)

    if T == 0:
        return []

    # An all-ON operational window has no complete OFF -> ON episode.
    # Preserve its observed duration; the absent state receives epsilon.
    if operational and np.all(X == X[0]):
        return [EpisodeT(
            start=index_offset,
            end=index_offset + T - 1,
            T_OFF=float(T) if X[0] == 0 else float(missing_duration),
            T_ON=float(T) if X[0] == 1 else float(missing_duration),
            timezone=timezone,
        )]

    # --------------------------------------------------
    # 1. Extract consecutive ON/OFF blocks
    # --------------------------------------------------
    runs = []

    run_start = 0
    current_state = X[0]

    for t in range(1, T):

        if X[t] != current_state:

            run_end = t - 1

            runs.append({
                "state": int(current_state),
                "start": int(run_start),
                "end": int(run_end),
                "duration": int(
                    run_end - run_start + 1
                ),
            })

            run_start = t
            current_state = X[t]

    # Close final block.
    run_end = T - 1

    runs.append({
        "state": int(current_state),
        "start": int(run_start),
        "end": int(run_end),
        "duration": int(
            run_end - run_start + 1
        ),
    })

    # --------------------------------------------------
    # 2. Identify complete OFF -> ON episodes
    # --------------------------------------------------
    complete_episodes = []

    for run_index in range(len(runs) - 1):

        off_run = runs[run_index]
        on_run = runs[run_index + 1]

        if (
            off_run["state"] == 0
            and on_run["state"] == 1
        ):
            complete_episodes.append(
                EpisodeT(
                    start=(
                        index_offset
                        + off_run["start"]
                    ),
                    end=(
                        index_offset
                        + on_run["end"]
                    ),
                    T_OFF=max(
                        minimum_duration,
                        off_run["duration"]
                    ),
                    T_ON=max(
                        minimum_duration,
                        on_run["duration"]
                    ),
                    timezone=timezone,
                )
            )

    if incomplete_policy == "nearest":
        return complete_episodes

    episodes = complete_episodes.copy()

    # The observed/one policies retain their existing boundary treatment.
    def complete_missing_duration(
        observed_duration,
        missing_state,
        local_start,
        local_end
    ):
        if incomplete_policy == "observed":
            return max(minimum_duration, observed_duration)
        return minimum_duration

    # --------------------------------------------------
    # 3. Initial ON block
    # --------------------------------------------------
    first_run = runs[0]

    if first_run["state"] == 1:

        observed_T_ON = max(
            minimum_duration,
            first_run["duration"]
        )

        estimated_T_OFF = (
            complete_missing_duration(
                observed_duration=observed_T_ON,
                missing_state=0,
                local_start=first_run["start"],
                local_end=first_run["end"]
            )
        )

        episodes.append(
            EpisodeT(
                start=(
                    index_offset
                    + first_run["start"]
                ),
                end=(
                    index_offset
                    + first_run["end"]
                ),
                T_OFF=int(estimated_T_OFF),
                T_ON=int(observed_T_ON),
                timezone=timezone,
            )
        )

    # --------------------------------------------------
    # 4. Final OFF block
    # --------------------------------------------------
    last_run = runs[-1]

    if last_run["state"] == 0:

        observed_T_OFF = max(
            minimum_duration,
            last_run["duration"]
        )

        estimated_T_ON = (
            complete_missing_duration(
                observed_duration=observed_T_OFF,
                missing_state=1,
                local_start=last_run["start"],
                local_end=last_run["end"]
            )
        )

        episodes.append(
            EpisodeT(
                start=(
                    index_offset
                    + last_run["start"]
                ),
                end=(
                    index_offset
                    + last_run["end"]
                ),
                T_OFF=int(observed_T_OFF),
                T_ON=int(estimated_T_ON),
                timezone=timezone,
            )
        )

    # Keep chronological order.
    episodes.sort(
        key=lambda episode: episode.start
    )

    return episodes

def validate_binary_profile(X):
    """
    Validate and return a one-dimensional binary profile.
    """

    X = np.asarray(X, dtype=int)

    if X.ndim != 1:
        raise ValueError(
            "X must be one-dimensional."
        )

    if X.size == 0:
        raise ValueError(
            "X cannot be empty."
        )

    if not np.all(np.isin(X, [0, 1])):
        raise ValueError(
            "X must contain only 0 and 1."
        )

    return X


In [ ]:
def compute_ton_toff(
    X,
    index_offset=0,
    incomplete_policy="observed",
    minimum_duration=1,
    operational=False,
    missing_duration=1e-6
):
    """
    Compute T_ON and T_OFF on the daily or operational horizon.

    Operational devices have float durations and NaN outside the first-to-
    last ON window. With no activations, both vectors are entirely NaN.
    A single-state window uses missing_duration for the absent state.

    With nearest, initial ON and final OFF stretches inherit BOTH
    durations from the closest complete OFF -> ON episode. The observed
    ON/OFF profile is preserved. A nonempty profile with no complete
    episode has no nearest reference and raises ValueError for non-operational
    devices. The single-state operational fallback applies to all policies.

    Parameters
    ----------
    X : ndarray
        Binary ON/OFF trajectory.

    Returns
    -------
    T_ON : ndarray
        ON duration assigned to every sample.

    T_OFF : ndarray
        OFF duration assigned to every sample.
    """

    episodes = extract_off_on_episodes(
        X=X,
        index_offset=index_offset,
        incomplete_policy=incomplete_policy,
        minimum_duration=minimum_duration,
        operational=operational,
        missing_duration=missing_duration,
    )

    T = len(X)

    window = get_operation_window(X) if operational else slice(0, T)
    fill_value = np.nan if operational else 0.0
    T_ON = np.full(T, fill_value, dtype=float)
    T_OFF = np.full(T, fill_value, dtype=float)

    for episode in episodes:

        start = episode.start - index_offset
        end = episode.end - index_offset + 1

        T_ON[start:end] = episode.T_ON
        T_OFF[start:end] = episode.T_OFF

    if incomplete_policy == "nearest" and window.stop > window.start:
        if not episodes:
            raise ValueError(
                "Cannot assign nearest T_ON/T_OFF: the profile contains "
                "no complete OFF -> ON episode."
            )

        first = episodes[0]
        initial_end = first.start - index_offset
        T_ON[window.start:initial_end] = first.T_ON
        T_OFF[window.start:initial_end] = first.T_OFF

        last = episodes[-1]
        final_start = last.end - index_offset + 1
        T_ON[final_start:window.stop] = last.T_ON
        T_OFF[final_start:window.stop] = last.T_OFF

    return T_ON, T_OFF

In [ ]:
def compute_time_weight(
    X,
    T_ON,
    T_OFF,
    W_min=1.0,
    W_max=10.0,
    clip=False
):
    """
    Compute the time-flexibility weight.

    During OFF:

        W[t] = W[t-1] + (W_max - W_min) / T_OFF[t]

    During ON:

        W[t] = W[t-1] - (W_max - W_min) / T_ON[t]

    Parameters
    ----------
    X : array-like, shape (T,)
        Binary ON/OFF trajectory.

    T_ON : array-like, shape (T,)
        ON durations.

    T_OFF : array-like, shape (T,)
        OFF durations.

    W_min : float, default=1.0
        Lower reference weight.

    W_max : float, default=10.0
        Upper reference weight.

    clip : bool, default=False
        If True, constrain W to [W_min, W_max].

    Returns
    -------
    W : ndarray of float, shape (T,)
        Time-flexibility weight.
    """

    X = validate_binary_profile(X)

    T_ON = np.asarray(T_ON, dtype=float)
    T_OFF = np.asarray(T_OFF, dtype=float)

    if T_ON.shape != X.shape:
        raise ValueError(
            "T_ON must have the same shape as X."
        )

    if T_OFF.shape != X.shape:
        raise ValueError(
            "T_OFF must have the same shape as X."
        )

    if W_max <= W_min:
        raise ValueError(
            "W_max must be greater than W_min."
        )

    if np.any(T_ON <= 0):
        raise ValueError(
            "T_ON must contain only positive values."
        )

    if np.any(T_OFF <= 0):
        raise ValueError(
            "T_OFF must contain only positive values."
        )

    delta_W = W_max - W_min

    W = np.empty(len(X), dtype=float)

    # Initialization
    if X[0] == 0:
        W[0] = W_min + delta_W / T_OFF[0]
    else:
        W[0] = W_max - delta_W / T_ON[0]

    if clip:
        W[0] = np.clip(
            W[0],
            W_min,
            W_max
        )

    # Evolution
    for t in range(1, len(X)):

        if X[t] == 0:
            W[t] = (
                W[t - 1]
                + delta_W / T_OFF[t]
            )
        else:
            W[t] = (
                W[t - 1]
                - delta_W / T_ON[t]
            )

        if clip:
            W[t] = np.clip(
                W[t],
                W_min,
                W_max
            )

    return W

def compute_power_weight(Y, Y_min, Y_max, W_P_min, W_P_max, m=1e-6):
    Y_safe = np.clip(Y, Y_min, Y_max)

    WP = W_P_min + ((W_P_max - W_P_min) * (Y_safe - Y_min)) / (
        (Y_max - Y_min) + m
    )

    return np.clip(WP, min(W_P_min, W_P_max), max(W_P_min, W_P_max))


In [ ]:
T_ON_FC_matrix = np.zeros((T, N_DEV),dtype=float)
T_OFF_FC_matrix = np.zeros((T, N_DEV),dtype=float)
WT_FC_matrix = np.zeros((T, N_DEV),dtype=float)
WP_FC_matrix = np.zeros((T, N_DEV),dtype=float)
WNF_FC_matrix = np.zeros((T, N_DEV),dtype=float)


episodes_by_device = {}
characterization_windows = {}
time_flex_available_matrix = np.zeros((T, N_DEV), dtype=bool)
time_flex_start_matrix = np.zeros((T, N_DEV), dtype=bool)

for j in range(N_DEV):

    if Type[j] == 2:
        X_j = X_onoff_FC_matrix[:, j]

        operational = j in OPERATIONAL_DEVICES
        window = get_operation_window(X_j) if operational else slice(0, T)
        characterization_windows[j] = window
        time_flex_available_matrix[window, j] = True
        if operational and window.stop > window.start:
            time_flex_start_matrix[window.start, j] = True

        episodes_by_device[j] = extract_off_on_episodes(
            X_j,
            incomplete_policy=INCOMPLETE_POLICY,
            operational=operational,
            missing_duration=MISSING_STATE_DURATION,
        )
        T_ON_j, T_OFF_j = compute_ton_toff(
            X_j,
            incomplete_policy=INCOMPLETE_POLICY,
            operational=operational,
            missing_duration=MISSING_STATE_DURATION,
        )
        T_ON_FC_matrix[:, j] = T_ON_j
        T_OFF_FC_matrix[:, j] = T_OFF_j

        WT_FC_matrix[:, j] = np.nan
        if window.stop > window.start:
            WT_FC_matrix[window, j] = compute_time_weight(
                X=X_j[window],
                T_ON=T_ON_j[window],
                T_OFF=T_OFF_j[window],
                clip=False,
            )
    
    elif Type[j] == 1:  # power-flexible
            WP_j = compute_power_weight(
                Y_FC_matrix[:, j],
                Y_min[j],
                Y_max[j],
                Weight_P_min[j],
                Weight_P_max[j]
            )
            WP_FC_matrix[:, j] = np.where(
                X_onoff_FC_matrix[:, j] == 1,
                WP_j,
                np.nan
            )
    elif Type[j] == 0:  # power-flexible
            WNF_FC_matrix[:, j]=10

# RT Control

#### Cenário a ser Testado

In [ ]:
# =====================================================
# RT INPUT FROM DAY-AHEAD PLANNING
# =====================================================

X_plan_matrix = X_onoff_FC_matrix.copy()
Y_plan_matrix = Y_FC_matrix.copy()

X_plan_ref_matrix = X_onoff_FC_matrix.copy()
Y_plan_ref_matrix = Y_FC_matrix.copy()

# Baseline power is retained only for diagnostics.
baseline_power_W = (
    X_plan_ref_matrix
    * Y_plan_ref_matrix
    * np.asarray(Power_max, dtype=float)[np.newaxis, :]
).sum(axis=1)
timezone_baseline_peaks_W = {
    tz: float(np.max(baseline_power_W[limits["start"]:limits["end"]]))
    for tz, limits in TIME_ZONES.items()
}
# RT uses the fixed service limit assigned to each timezone.
timezone_power_limits_W = TIMEZONE_POWER_LIMITS_W.copy()
power_limit_setup = pd.DataFrame({
    "Timezone": list(TIME_ZONES),
    "Baseline peak [kW]": [
        timezone_baseline_peaks_W[tz] / 1000 for tz in TIME_ZONES
    ],
    "Selected limit [kW]": [
        timezone_power_limits_W[tz] / 1000 for tz in TIME_ZONES
    ],
})
display(power_limit_setup.style.hide(axis="index").format(precision=3))

In [ ]:
# ============================================================
# MAX UTILITY — STORAGE
# ============================================================

X_utility_rt_matrix = np.zeros(
    (T, N_DEV),
    dtype=int
)
prevent_off_utility_rt_matrix = np.zeros(
    (T, N_DEV),
    dtype=int,
)

Y_utility_rt_matrix = np.ones(
    (T, N_DEV),
    dtype=float
)

W_T_utility_rt_matrix = np.zeros(
    (T, N_DEV),
    dtype=float
)

W_P_utility_rt_matrix = np.zeros(
    (T, N_DEV),
    dtype=float
)

W_utility_rt_matrix = np.zeros(
    (T, N_DEV),
    dtype=float
)

P_utility_rt_matrix = np.zeros(
    (T, N_DEV),
    dtype=float
)

P_utility_rt_sum = np.zeros(
    T,
    dtype=float
)


force_on_utility_rt_matrix = np.zeros(
    (T, N_DEV),
    dtype=int
)

count_hold_utility_matrix = np.zeros(
    (T, N_DEV),
    dtype=int
)


P_lim_utility_vector = np.zeros(
    T,
    dtype=float
)

P_rlx_utility_vector = np.zeros(
    T,
    dtype=float
)


P_before_utility_sum = np.zeros(
    T,
    dtype=float
)
force_off_utility_rt_matrix = np.zeros(
    (T, N_DEV),
    dtype=int
)
prevent_off_vector = np.zeros(
    N_DEV,
    dtype=int
)

# ============================================================
# MAX UTILITY + VARIABLE HYSTERESIS STORAGE
# ============================================================

X_utility_hyst_rt_matrix = np.zeros(
    (T, N_DEV),
    dtype=int
)

Y_utility_hyst_rt_matrix = np.ones(
    (T, N_DEV),
    dtype=float
)

W_utility_hyst_rt_matrix = np.zeros(
    (T, N_DEV),
    dtype=float
)
W_T_utility_hyst_rt_matrix = np.full(
    (T, N_DEV),
    np.nan,
    dtype=float
)

W_P_utility_hyst_rt_matrix = np.full(
    (T, N_DEV),
    np.nan,
    dtype=float
)

P_utility_hyst_rt_matrix = np.zeros(
    (T, N_DEV),
    dtype=float
)

P_utility_hyst_rt_sum = np.zeros(
    T,
    dtype=float
)

P_before_utility_hyst_sum = np.zeros(
    T,
    dtype=float
)

force_on_utility_hyst_rt_matrix = np.zeros(
    (T, N_DEV),
    dtype=int
)

force_off_utility_hyst_rt_matrix = np.zeros(
    (T, N_DEV),
    dtype=int
)

count_hold_utility_hyst_matrix = np.zeros(
    (T, N_DEV),
    dtype=int
)

P_lim_utility_hyst_vector = np.zeros(
    T,
    dtype=float
)

P_rlx_utility_hyst_vector = np.zeros(
    T,
    dtype=float
)


# Optional:
# Store the Wmax actually applicable at each minute/device
W_max_utility_hyst_matrix = np.zeros(
    (T, N_DEV),
    dtype=float
)

# ============================================================
# MIN ENERGY — STORAGE
# ============================================================

X_energy_rt_matrix = np.zeros(
    (T, N_DEV),
    dtype=int
)

Y_energy_rt_matrix = np.ones(
    (T, N_DEV),
    dtype=float
)

W_T_energy_rt_matrix = np.full(
    (T, N_DEV),
    np.nan,
    dtype=float
)

W_P_energy_rt_matrix = np.full(
    (T, N_DEV),
    np.nan,
    dtype=float
)

W_energy_rt_matrix = np.full(
    (T, N_DEV),
    np.nan,
    dtype=float
)

P_energy_rt_matrix = np.zeros(
    (T, N_DEV),
    dtype=float
)

P_energy_rt_sum = np.zeros(
    T,
    dtype=float
)


force_on_energy_rt_matrix = np.zeros(
    (T, N_DEV),
    dtype=int
)


force_off_energy_rt_matrix = np.zeros(
    (T, N_DEV),
    dtype=int
)

count_hold_energy_matrix = np.zeros(
    (T, N_DEV),
    dtype=int
)


P_lim_energy_vector = np.zeros(
    T,
    dtype=float
)

P_rlx_energy_vector = np.zeros(
    T,
    dtype=float
)

P_before_energy_sum = np.zeros(
    T,
    dtype=float
)

prevent_off_energy_rt_matrix = np.zeros(
    (T, N_DEV),
    dtype=int,
)

#### Funções Auxiliares RT Control

In [ ]:
def get_w_max(j, minute, variable_hysteresis=False):

    if variable_hysteresis:

        # Drinks Refrigerator
        if j == 0:

            # 13:00 -> 15:00
            if 22*60+30 <= minute < 24*60:
                return 20.0
    
    return 10.0

def function_dataset(
    i,
    X_RT_matrix,
    W_T_RT_matrix,
    X_plan_matrix,
    Type,
    time_flex_available_matrix,
    time_flex_start_matrix,
    variable_hysteresis=False
):

    X_RT_vector = X_plan_matrix[
        i,
        :
    ].copy()

    force_on_vector = np.zeros(
        len(Type),
        dtype=int
    )

    force_off_vector = np.zeros(
        len(Type),
        dtype=int
    )

    if i == 0:
        return (
            X_RT_vector,
            force_on_vector,
            force_off_vector
        )


    for j in range(len(Type)):

        if Type[j] == 2:

            # Preserve the reference outside the window. On entry there is
            # no previous in-window state to use for threshold overrides.
            if (not time_flex_available_matrix[i, j]
                    or time_flex_start_matrix[i, j]):
                continue

            previous_W = float(
                W_T_RT_matrix[
                    i - 1,
                    j
                ]
            )

            if not np.isfinite(previous_W):
                raise ValueError(f"Missing active RT weight at minute {i}, device {j}.")

            W_max_now = get_w_max(
                j,
                i,
                variable_hysteresis
            )


            # ============================================
            # MAXIMUM ON TIME REACHED
            # ============================================

            if previous_W <= 1.0:
                X_RT_vector[j] = 0
                force_off_vector[j] = 1
            # ============================================
            # MAXIMUM OFF TIME REACHED
            # ============================================
            elif previous_W >= W_max_now:
                X_RT_vector[j] = 1
                force_on_vector[j] = 1


            # ============================================
            # NO OPERATIONAL OVERRIDE
            # ============================================

            else:

                X_RT_vector[j] = (
                    X_plan_matrix[
                        i,
                        j
                    ]
                )


        else:

            X_RT_vector[j] = (
                X_plan_matrix[
                    i,
                    j
                ]
            )


    return (
        X_RT_vector,
        force_on_vector,
        force_off_vector
    )
    
def update_power_before_optimization(
    X_before_opt_rt_vector,
    Y_plan_vector,
    Y_prev_opt_rt_vector,
    count_prev_vector,
    Type,
):
    """
    Build Y before RT optimization.

    Type 1:
        Follow Day-Ahead Y_plan when no power-hold constraint is active.

        If still inside T_Hold_Power:
            keep previous executed Y.

    Other types:
        Y = 1.
    """

    Y_before_opt_rt_vector = np.ones(
        len(Type),
        dtype=float
    )

    count_vector = count_prev_vector.copy()

    for j in range(len(Type)):

        # Device is OFF
        if X_before_opt_rt_vector[j] == 0:

            Y_before_opt_rt_vector[j] = 1.0
            count_vector[j] = 0


        # ============================================
        # POWER-FLEXIBLE
        # ============================================

        elif Type[j] == 1:

            if count_vector[j] > 0:

                # Still within power hold
                Y_before_opt_rt_vector[j] = (
                    Y_prev_opt_rt_vector[j]
                )

                count_vector[j] -= 1

            else:

                # Follow Day-Ahead power recommendation
                Y_before_opt_rt_vector[j] = (
                    Y_plan_vector[j]
                )


        # ============================================
        # TYPE 0 / TYPE 2
        # ============================================

        else:

            Y_before_opt_rt_vector[j] = 1.0
            count_vector[j] = 0


    return (
        Y_before_opt_rt_vector,
        count_vector
    )

def update_power_hold_after_optimization(
    X_opt_rt_vector,
    Y_before_opt_rt_vector,
    Y_opt_rt_vector,
    count_vector,
    Type,
    TimeHoldPower,
):
    count_updated = count_vector.copy()

    for j in range(len(Type)):

        if Type[j] == 1 and X_opt_rt_vector[j] == 1:

            if abs(Y_opt_rt_vector[j] - Y_before_opt_rt_vector[j]) > 1e-6:
                count_updated[j] = int(TimeHoldPower[j])

        else:
            count_updated[j] = 0

    return count_updated

def time_weight_delta(state, t_on, t_off):
    """One-minute weight change; preserve positive sub-minute durations."""
    if not (np.isfinite(t_on) and np.isfinite(t_off)
            and t_on > 0 and t_off > 0):
        raise ValueError("Active time-flexible durations must be finite and positive.")
    return -9.0 / t_on if state == 1 else 9.0 / t_off


def initialize_time_weights_from_state(
    X0,
    Type,
    T_ON_0,
    T_OFF_0,
    time_flex_available,
):
    W_T0 = np.full(len(X0), np.nan, dtype=float)

    for j in range(len(X0)):
        if Type[j] != 2 or not time_flex_available[j]:
            continue

        W_T0[j] = 10.0 if X0[j] == 1 else 1.0

    return W_T0


def prepare_time_weights_for_minute(
    prev_W_T,
    X_reference,
    Type,
    time_flex_available,
    time_flex_start,
):
    """Prepare pre-decision weights, seeding each operational window once.

    Availability comes from the reference window, not from NaN comparisons.
    A missing weight inside an ongoing window is an error.
    """
    weights = np.full(len(Type), np.nan, dtype=float)
    for j in range(len(Type)):
        if Type[j] != 2 or not time_flex_available[j]:
            continue
        if time_flex_start[j]:
            weights[j] = 10.0 if X_reference[j] == 1 else 1.0
        elif np.isfinite(prev_W_T[j]):
            weights[j] = prev_W_T[j]
        else:
            raise ValueError(f"Missing weight inside active window for device {j}.")
    return weights

def compute_power_vector(X_vec, Power_max, Y_vec):
    """
    Compute device power vector.

    P[j] = Power_max[j] * Y[j] * X[j]
    """
    return Power_max * Y_vec * X_vec

def compute_power_sum(P_vec):
    """
    Total power at the current minute.
    """
    return float(np.sum(P_vec))

def update_time_weights(
    prev_W_T,
    X_now,
    Type,
    T_ON_now,
    T_OFF_now,
    minute,
    time_flex_available,
    variable_hysteresis=False,
):
    """Update available time weights without clipping or duration rounding.

    Outside the operating window weights remain NaN. Within it, reactive
    thresholds are still applied by function_dataset on the next minute.
    minute and variable_hysteresis retain their existing interface role.
    """
    W_T_new = np.full(len(X_now), np.nan, dtype=float)
    for j in range(len(X_now)):
        if Type[j] != 2 or not time_flex_available[j]:
            continue
        if not np.isfinite(prev_W_T[j]):
            raise ValueError(f"Missing active weight at minute {minute}, device {j}.")
        W_T_new[j] = prev_W_T[j] + time_weight_delta(
            X_now[j], T_ON_now[j], T_OFF_now[j]
        )
    return W_T_new
def compute_wp_vector(Y_vec, Weight_P_min, Weight_P_max, Y_min, Y_max):
    epsilon = 1e-4

    return (
        Weight_P_min
        +
        (
            (Weight_P_max - Weight_P_min)
            /
            (Y_max - Y_min + epsilon)
        )
        * (Y_vec - Y_min)
    )

## Modelos

#### Modelo Optimização - Max Utility

In [ ]:
m = pe.ConcreteModel()

# =========================
# Sets
# =========================
m.J = pe.RangeSet(0, N_DEV - 1)

# =========================
# Fixed parameters
# =========================
m.Type = pe.Param(m.J, initialize={j: int(Type[j]) for j in range(N_DEV)})
m.P_max = pe.Param(m.J, initialize={j: float(Power_max[j]) for j in range(N_DEV)})

m.Y_min = pe.Param(m.J, initialize={j: float(Y_min[j]) for j in range(N_DEV)})
m.Y_max = pe.Param(m.J, initialize={j: float(Y_max[j]) for j in range(N_DEV)})


# =========================
# Mutable RT parameters
# updated each minute
# =========================
m.P_lim = pe.Param(initialize=0.0, mutable=True)

m.X_in = pe.Param(m.J, initialize=0, mutable=True)
m.W_time = pe.Param(m.J, initialize=0.0, mutable=True, within=pe.Reals)
m.W_power = pe.Param(m.J,initialize=1.0,mutable=True)


# Y actual antes da otimização
m.Y_in = pe.Param(m.J, initialize={j: float(Y_min[j]) for j in range(N_DEV)}, mutable=True)

# contador do tempo que falta / já passou desde alteração de potência
m.count_T_Hold_Power = pe.Param(m.J, initialize=0, mutable=True)

m.force_on = pe.Param(m.J, initialize=0, mutable=True)
m.force_off = pe.Param(m.J, initialize=0, mutable=True)
m.time_flex_available = pe.Param(m.J, initialize=0, mutable=True, within=pe.Binary)

m.prevent_off = pe.Param(m.J,initialize=0,mutable=True,within=pe.Binary)

# =========================
# Decision variables
# =========================
m.X_opt = pe.Var(m.J, domain=pe.Binary)
m.Y_opt = pe.Var(m.J, domain=pe.NonNegativeReals)
m.Y_active = pe.Var(m.J, domain=pe.NonNegativeReals)
m.P_rlx = pe.Var(domain=pe.NonNegativeReals)

# =========================
# Constraints
# =========================
# Total power constraint
def power_limit_rule(m):
    return sum(
        m.P_max[j] * m.Y_active[j]
        for j in m.J
    ) <= m.P_lim + m.P_rlx

m.power_limit = pe.Constraint(rule=power_limit_rule)

def y_active_upper_x_rule(m, j):
    return m.Y_active[j] <= m.Y_max[j] * m.X_opt[j]

def y_active_upper_y_rule(m, j):
    return m.Y_active[j] <= m.Y_opt[j]

def y_active_lower_rule(m, j):
    return m.Y_active[j] >= (
        m.Y_opt[j] - m.Y_max[j] * (1 - m.X_opt[j])
    )

m.y_active_upper_x = pe.Constraint(m.J, rule=y_active_upper_x_rule)
m.y_active_upper_y = pe.Constraint(m.J, rule=y_active_upper_y_rule)
m.y_active_lower = pe.Constraint(m.J, rule=y_active_lower_rule)


# Y lower bound
def y_min_rule(m, j):
    return m.Y_opt[j] >= m.Y_min[j]

m.y_min_constraint = pe.Constraint(m.J, rule=y_min_rule)


# Y upper bound
def y_max_rule(m, j):
    return m.Y_opt[j] <= m.Y_max[j]

m.y_max_constraint = pe.Constraint(m.J, rule=y_max_rule)

def prevent_weight_overshoot_rule(m, j):

    if pe.value(m.Type[j]) != 2:
        return pe.Constraint.Skip

    return m.X_opt[j] >= m.prevent_off[j]


m.prevent_weight_overshoot = pe.Constraint(
    m.J,
    rule=prevent_weight_overshoot_rule
)

# ---------------------------
# Constraint 2:
# Status logic by flexibility type
#
# Type 0 = non-flexible  -> X_opt = X_in
# Type 1 = power-flexible -> X_opt = X_in
# Type 2 = time-flexible  -> X_opt <= X_in
#
# Outside a time-flexibility window: X_opt = X_in
# ---------------------------
def status_logic_rule(m, j):

    device_type = int(
        pe.value(m.Type[j])
    )

    if device_type == 2:

        # Fora da janela de flexibilidade:
        # seguir a referência.
        if int(
            pe.value(m.time_flex_available[j])
        ) == 0:

            return (
                m.X_opt[j]
                ==
                m.X_in[j]
            )

        # Tempo máximo OFF atingido.
        if int(pe.value(m.force_on[j])) == 1:
            return (
                m.X_opt[j]
                ==
                1
            )

        # Tempo máximo ON atingido.
        if int(
            pe.value(m.force_off[j])
        ) == 1:

            return (
                m.X_opt[j]
                ==
                0
            )

        # Dentro da janela, o RT pode ligar ou desligar.
        return pe.Constraint.Feasible

    if device_type in [0, 1]:

        return (
            m.X_opt[j]
            ==
            m.X_in[j]
        )

    return pe.Constraint.Feasible
m.status_logic = pe.Constraint(
    m.J,
    rule=status_logic_rule
)

def y_rt_upper_rule(m, j):

    if pe.value(m.Type[j]) == 1:
        return m.Y_opt[j] <= m.Y_in[j]

    return pe.Constraint.Skip


m.y_rt_upper = pe.Constraint(
    m.J,
    rule=y_rt_upper_rule
)

def power_hold_rule(m, j):
    # se não for power-flexible, mantém Y = Y_in
    if pe.value(m.Type[j]) != 1:
        return m.Y_opt[j] == m.Y_in[j]

    # se ainda está dentro do T_Hold_Power, mantém o Y anterior
    if pe.value(m.count_T_Hold_Power[j]) != 0:
        return m.Y_opt[j] == m.Y_in[j]

    # caso contrário, pode otimizar Y dentro de [Y_min, Y_max]
    return pe.Constraint.Feasible

m.power_hold_constraint = pe.Constraint(m.J, rule=power_hold_rule)

# ============================================================
# OBJECTIVE — MAXIMIZE UTILITY
# ============================================================

lambda_T_RT = 1.0
lambda_P_RT = 1.0
lambda_rlx = 1000.0

W_max_RT = 10.0


def objective_rule(m):

    # ========================================================
    # TIME-FLEXIBLE UTILITY LOSS
    # ========================================================

    time_utility_loss = sum(

        # Initially ON:
        # switching OFF has cost W_T
        m.W_time[j]
        * m.X_in[j]
        * (1 - m.X_opt[j])

        +

        # Initially OFF:
        # switching ON has cost W_max - W_T
        (W_max_RT - m.W_time[j])
        * (1 - m.X_in[j])
        * m.X_opt[j]

        for j in m.J

        if int(pe.value(m.Type[j])) == 2
    )

    # ========================================================
    # POWER-FLEXIBLE UTILITY LOSS
    # ========================================================

    power_utility_loss = sum(

        m.W_power[j]
        * (
            m.Y_in[j]
            - m.Y_opt[j]
        )

        for j in m.J

        if int(pe.value(m.Type[j])) == 1
    )

    # ========================================================
    # OBJECTIVE
    # ========================================================

    return (

        lambda_T_RT
        * time_utility_loss

        +

        lambda_P_RT
        * power_utility_loss

        +

        lambda_rlx
        * m.P_rlx
    )


m.objective = pe.Objective(
    rule=objective_rule,
    sense=pe.minimize
)


In [ ]:
# ============================================================
# MAX UTILITY RT LOOP
# ============================================================

from time import perf_counter

# Wall-clock time per control iteration; first iteration is included.
rt_iteration_time_utility_s = np.full(T, np.nan)
rt_solver_time_utility_s = np.full(T, np.nan)

solver_utility = pe.SolverFactory("highs")

variable_hysteresis = False

# ============================================================
# INITIAL STATE
# ============================================================

X_initial = X_onoff_FC_matrix[0, :].copy()

W_T_initial = initialize_time_weights_from_state(
    X0=X_initial,
    Type=Type,
    T_ON_0=T_ON_FC_matrix[0, :],
    T_OFF_0=T_OFF_FC_matrix[0, :],
    time_flex_available=time_flex_available_matrix[0, :],
)

count_initial = np.zeros(
    N_DEV,
    dtype=int
)

# ============================================================
# MINUTE-BY-MINUTE RT CONTROL
# ============================================================

for i in range(T):

    rt_iteration_started = perf_counter()

    # ========================================================
    # STEP 1 — CURRENT INPUTS
    # ========================================================

    P_lim = get_p_lim_for_minute(i)

    T_ON_now = T_ON_FC_matrix[i, :].copy()
    T_OFF_now = T_OFF_FC_matrix[i, :].copy()
    time_flex_available_now = time_flex_available_matrix[i, :]

    X_plan_vector = X_plan_ref_matrix[i, :].copy()
    Y_plan_vector = Y_plan_ref_matrix[i, :].copy()

    P_lim_utility_vector[i] = P_lim

    # ========================================================
    # STEP 2 — PREVIOUS RT STATE
    # ========================================================

    if i == 0:

        X_previous = X_initial.copy()

        Y_previous = np.ones(
            N_DEV,
            dtype=float
        )

        W_T_previous = W_T_initial.copy()

        count_previous = count_initial.copy()

    else:

        X_previous = X_utility_rt_matrix[
            i - 1, :
        ].copy()

        Y_previous = Y_utility_rt_matrix[
            i - 1, :
        ].copy()

        W_T_previous = W_T_utility_rt_matrix[
            i - 1, :
        ].copy()

        count_previous = count_hold_utility_matrix[
            i - 1, :
        ].copy()

    W_T_previous = prepare_time_weights_for_minute(
        prev_W_T=W_T_previous,
        X_reference=X_plan_vector,
        Type=Type,
        time_flex_available=time_flex_available_now,
        time_flex_start=time_flex_start_matrix[i, :],
    )

    # ========================================================
    # STEP 3 — TIME-FLEXIBLE PRE-PROCESSING
    #
    # Build X_before and operational overrides.
    # ========================================================

    if i == 0:

        X_before = X_plan_vector.copy()

        force_on_vector = np.zeros(
            N_DEV,
            dtype=int
        )

        force_off_vector = np.zeros(
            N_DEV,
            dtype=int
        )

    else:

        (
            X_before,
            force_on_vector,
            force_off_vector
        ) = function_dataset(

            i=i,

            X_RT_matrix=X_utility_rt_matrix,

            W_T_RT_matrix=W_T_utility_rt_matrix,

            X_plan_matrix=X_plan_ref_matrix,

            Type=Type,

            time_flex_available_matrix=time_flex_available_matrix,
            time_flex_start_matrix=time_flex_start_matrix,
            variable_hysteresis=variable_hysteresis
        )

    # ========================================================
    # STEP 4 — POWER-FLEXIBLE PRE-PROCESSING
    # ========================================================

    (
        Y_before,
        count_vector
    ) = update_power_before_optimization(

        X_before_opt_rt_vector=X_before,

        Y_plan_vector=Y_plan_vector,

        Y_prev_opt_rt_vector=Y_previous,

        count_prev_vector=count_previous,

        Type=Type
    )

    # ========================================================
    # STEP 5 — POWER-FLEXIBILITY WEIGHT
    # ========================================================

    W_P_before = compute_wp_vector(

        Y_vec=Y_before,

        Weight_P_min=Weight_P_min,

        Weight_P_max=Weight_P_max,

        Y_min=Y_min,

        Y_max=Y_max
    )

    # ========================================================
    # OPTIONAL — POWER BEFORE RT OPTIMIZATION
    # ========================================================

    P_before = compute_power_vector(
        X_vec=X_before,
        Power_max=Power_max,
        Y_vec=Y_before
    )

    P_before_utility_sum[i] = compute_power_sum(
        P_before
    )
    
    prevent_off_vector = np.zeros(
        N_DEV,
        dtype=int
    )

    for j in range(N_DEV):

        if (
            Type[j] == 2
            and time_flex_available_now[j]
            and X_before[j] == 1
            and force_on_vector[j] == 0
            and force_off_vector[j] == 0
        ):
            W_if_off = (
                W_T_previous[j]
                + 9.0 / T_OFF_now[j]
            )

            W_max_now = get_w_max(
                j,
                i,
                variable_hysteresis
            )

            prevent_off_vector[j] = int(
                W_if_off >= W_max_now - 1e-9
            )

    # ========================================================
    # STEP 6 — UPDATE OPTIMIZATION MODEL
    # ========================================================

    m.P_lim.set_value(
        float(P_lim)
    )

    for j in range(N_DEV):

        m.time_flex_available[j] = int(time_flex_available_now[j])

        # Never send NaN to the solver. Unavailable states are fixed.
        if Type[j] == 2 and time_flex_available_now[j]:
            m.W_time[j] = float(
                W_T_previous[j]
            )
        else:
            m.W_time[j] = 0.0

        # Power-flexibility utility weight
        if Type[j] == 1:
            m.W_power[j] = float(
                W_P_before[j]
            )
        else:
            m.W_power[j] = 0.0
            
        m.prevent_off[j] = int(prevent_off_vector[j])
        
        # State before optimization
        m.X_in[j] = int(
            X_before[j]
        )

        m.Y_in[j] = float(
            Y_before[j]
        )

        # Operational constraints
        m.force_on[j] = int(
            force_on_vector[j]
        )

        m.force_off[j] = int(
            force_off_vector[j]
        )

        # Power hold
        m.count_T_Hold_Power[j] = int(
            count_vector[j]
        )

        # Solver initial values
        m.X_opt[j].value = int(
            X_before[j]
        )

        m.Y_opt[j].value = float(
            Y_before[j]
        )

    m.P_rlx.value = 0.0

    # ========================================================
    # STEP 7 — REBUILD DYNAMIC CONSTRAINTS
    # ========================================================

    if hasattr(m, "status_logic"):
        m.del_component(
            m.status_logic
        )

    if hasattr(
        m,
        "power_hold_constraint"
    ):
        m.del_component(
            m.power_hold_constraint
        )

    m.status_logic = pe.Constraint(
        m.J,
        rule=status_logic_rule
    )

    m.power_hold_constraint = pe.Constraint(
        m.J,
        rule=power_hold_rule
    )

    # ========================================================
    # STEP 8 — REAL-TIME OPTIMIZATION
    # ========================================================

    rt_solver_started = perf_counter()
    results = solver_utility.solve(
        m,
        tee=False,
        load_solutions=False
    )

    rt_solver_time_utility_s[i] = perf_counter() - rt_solver_started

    termination = (
        results.solver.termination_condition
    )

    if (
        termination
        ==
        pe.TerminationCondition.optimal
    ):

        m.solutions.load_from(
            results
        )

    else:

        raise RuntimeError(
            f"Max Utility optimization failed "
            f"at minute {i}: "
            f"{termination}"
        )

    # ========================================================
    # STEP 9 — EXECUTED RT DECISION
    # ========================================================

    X_final = np.zeros(
        N_DEV,
        dtype=int
    )

    Y_final = np.ones(
        N_DEV,
        dtype=float
    )

    for j in range(N_DEV):

        X_final[j] = int(
            round(
                pe.value(
                    m.X_opt[j]
                )
            )
        )

        if X_final[j] == 1:

            Y_final[j] = float(
                pe.value(
                    m.Y_opt[j]
                )
            )

        else:

            Y_final[j] = 1.0

    # ========================================================
    # STEP 10 — UPDATE POWER HOLD
    # ========================================================

    count_vector = (
        update_power_hold_after_optimization(

            X_opt_rt_vector=X_final,

            Y_before_opt_rt_vector=Y_before,

            Y_opt_rt_vector=Y_final,

            count_vector=count_vector,

            Type=Type,

            TimeHoldPower=TimeHoldPower
        )
    )

    # ========================================================
    # STEP 11 — FINAL RT STATES
    # ========================================================

    P_rlx_utility_vector[i] = float(
        pe.value(
            m.P_rlx
        )
    )

    P_final = compute_power_vector(

        X_vec=X_final,

        Power_max=Power_max,

        Y_vec=Y_final
    )

    W_T_final = update_time_weights(

        prev_W_T=W_T_previous,

        X_now=X_final,

        Type=Type,

        T_ON_now=T_ON_now,

        T_OFF_now=T_OFF_now,

        minute=i,

        time_flex_available=time_flex_available_now,

        variable_hysteresis=variable_hysteresis
    )

    W_P_final = compute_wp_vector(

        Y_vec=Y_final,

        Weight_P_min=Weight_P_min,

        Weight_P_max=Weight_P_max,

        Y_min=Y_min,

        Y_max=Y_max
    )

    # ========================================================
    # STEP 12 — COMBINED FLEXIBILITY WEIGHT
    # ========================================================

    W_combined_final = np.full(
        N_DEV,
        np.nan,
        dtype=float
    )

    for j in range(N_DEV):

        if Type[j] == 1:
            W_combined_final[j] = W_P_final[j]

        elif Type[j] == 2:
            W_combined_final[j] = W_T_final[j]

    # ========================================================
    # STEP 13 — STORE RT STATE
    # ========================================================
    
    prevent_off_utility_rt_matrix[i, :] = prevent_off_vector

    X_utility_rt_matrix[i, :] = X_final

    Y_utility_rt_matrix[i, :] = Y_final

    W_T_utility_rt_matrix[i, :] = W_T_final

    W_P_utility_rt_matrix[i, :] = W_P_final

    W_utility_rt_matrix[i, :] = W_combined_final

    P_utility_rt_matrix[i, :] = P_final

    P_utility_rt_sum[i] = compute_power_sum(
        P_final
    )

    force_on_utility_rt_matrix[
        i, :
    ] = force_on_vector

    force_off_utility_rt_matrix[
        i, :
    ] = force_off_vector

    count_hold_utility_matrix[
        i, :
    ] = count_vector

    rt_iteration_time_utility_s[i] = perf_counter() - rt_iteration_started


#### Modelo Optimização - Min Energy Consumption

In [ ]:
# ============================================================
# MIN ENERGY CONSUMPTION RT MODEL
# ============================================================

m_energy = pe.ConcreteModel()


# ============================================================
# SETS
# ============================================================

m_energy.J = pe.RangeSet(
    0,
    N_DEV - 1
)


# ============================================================
# FIXED PARAMETERS
# ============================================================

m_energy.Type = pe.Param(
    m_energy.J,
    initialize={
        j: int(Type[j])
        for j in range(N_DEV)
    }
)

m_energy.P_max = pe.Param(
    m_energy.J,
    initialize={
        j: float(Power_max[j])
        for j in range(N_DEV)
    }
)

m_energy.Y_min = pe.Param(
    m_energy.J,
    initialize={
        j: float(Y_min[j])
        for j in range(N_DEV)
    }
)

m_energy.Y_max = pe.Param(
    m_energy.J,
    initialize={
        j: float(Y_max[j])
        for j in range(N_DEV)
    }
)


# ============================================================
# MUTABLE RT PARAMETERS
# ============================================================

m_energy.P_lim = pe.Param(
    initialize=0.0,
    mutable=True
)

m_energy.X_in = pe.Param(
    m_energy.J,
    initialize=0,
    mutable=True
)

m_energy.Y_in = pe.Param(
    m_energy.J,
    initialize={
        j: float(Y_min[j])
        for j in range(N_DEV)
    },
    mutable=True
)

m_energy.count_T_Hold_Power = pe.Param(
    m_energy.J,
    initialize=0,
    mutable=True
)


# ============================================================
# OPERATIONAL FLAGS
# ============================================================

m_energy.force_on = pe.Param(
    m_energy.J,
    initialize=0,
    mutable=True
)

m_energy.force_off = pe.Param(
    m_energy.J,
    initialize=0,
    mutable=True
)
m_energy.time_flex_available = pe.Param(
    m_energy.J, initialize=0, mutable=True, within=pe.Binary
)
m_energy.prevent_off = pe.Param(
    m_energy.J, initialize=0, mutable=True, within=pe.Binary
)


# ============================================================
# DECISION VARIABLES
# ============================================================

m_energy.X_opt = pe.Var(
    m_energy.J,
    domain=pe.Binary
)

m_energy.Y_opt = pe.Var(
    m_energy.J,
    domain=pe.NonNegativeReals
)

m_energy.Y_active = pe.Var(
    m_energy.J,
    domain=pe.NonNegativeReals
)

m_energy.P_rlx = pe.Var(
    domain=pe.NonNegativeReals
)


# ============================================================
# POWER LIMIT
# ============================================================

def energy_power_limit_rule(m):

    return sum(
        m.P_max[j]
        *
        m.Y_active[j]

        for j in m.J
    ) <= (
        m.P_lim
        +
        m.P_rlx
    )


m_energy.power_limit = pe.Constraint(
    rule=energy_power_limit_rule
)

def energy_y_active_upper_x_rule(m, j):
    return m.Y_active[j] <= m.Y_max[j] * m.X_opt[j]

def energy_y_active_upper_y_rule(m, j):
    return m.Y_active[j] <= m.Y_opt[j]

def energy_y_active_lower_rule(m, j):
    return m.Y_active[j] >= (
        m.Y_opt[j] - m.Y_max[j] * (1 - m.X_opt[j])
    )

m_energy.y_active_upper_x = pe.Constraint(
    m_energy.J,
    rule=energy_y_active_upper_x_rule
)
m_energy.y_active_upper_y = pe.Constraint(
    m_energy.J,
    rule=energy_y_active_upper_y_rule
)
m_energy.y_active_lower = pe.Constraint(
    m_energy.J,
    rule=energy_y_active_lower_rule
)

# ============================================================
# Y BOUNDS
# ============================================================

def energy_y_min_rule(m, j):

    return (
        m.Y_opt[j]
        >=
        m.Y_min[j]
    )


def energy_y_max_rule(m, j):

    return (
        m.Y_opt[j]
        <=
        m.Y_max[j]
    )


m_energy.y_min_constraint = pe.Constraint(
    m_energy.J,
    rule=energy_y_min_rule
)

m_energy.y_max_constraint = pe.Constraint(
    m_energy.J,
    rule=energy_y_max_rule
)


def energy_prevent_weight_overshoot_rule(m, j):

    if pe.value(m.Type[j]) != 2:
        return pe.Constraint.Skip

    return m.X_opt[j] >= m.prevent_off[j]


m_energy.prevent_weight_overshoot = pe.Constraint(
    m_energy.J,
    rule=energy_prevent_weight_overshoot_rule
)

# ============================================================
# STATUS LOGIC
# ============================================================

def energy_status_logic_rule(m, j):

    device_type = int(
        pe.value(m.Type[j])
    )

    if device_type in [0, 1]:

        return (
            m.X_opt[j]
            ==
            m.X_in[j]
        )

    if device_type == 2:

        # Fora da janela de flexibilidade:
        # seguir a referência.
        if int(
            pe.value(m.time_flex_available[j])
        ) == 0:

            return (
                m.X_opt[j]
                ==
                m.X_in[j]
            )

        # Tempo máximo OFF atingido.
        if int(
            pe.value(m.force_on[j])
        ) == 1:

            return (
                m.X_opt[j]
                ==
                1
            )

        # Tempo máximo ON atingido.
        if int(
            pe.value(m.force_off[j])
        ) == 1:

            return (
                m.X_opt[j]
                ==
                0
            )

        # Dentro da janela, o RT pode ligar ou desligar.
        return pe.Constraint.Feasible

    return pe.Constraint.Feasible


m_energy.status_logic = pe.Constraint(
    m_energy.J,
    rule=energy_status_logic_rule
)


# ============================================================
# TYPE 1:
# Y CAN ONLY BE REDUCED
# ============================================================

def energy_y_rt_upper_rule(m, j):

    if int(
        pe.value(
            m.Type[j]
        )
    ) == 1:

        return (
            m.Y_opt[j]
            <=
            m.Y_in[j]
        )

    return pe.Constraint.Skip


m_energy.y_rt_upper = pe.Constraint(
    m_energy.J,
    rule=energy_y_rt_upper_rule
)


# ============================================================
# POWER HOLD
# ============================================================

def energy_power_hold_rule(m, j):

    device_type = int(
        pe.value(
            m.Type[j]
        )
    )


    # Type 0 / Type 2:
    # Y does not change
    if device_type != 1:

        return (
            m.Y_opt[j]
            ==
            m.Y_in[j]
        )


    # Type 1:
    # existing hold constraint
    if int(
        pe.value(
            m.count_T_Hold_Power[j]
        )
    ) != 0:

        return (
            m.Y_opt[j]
            ==
            m.Y_in[j]
        )


    # Otherwise power-flexible device may reduce Y
    return pe.Constraint.Feasible


m_energy.power_hold_constraint = pe.Constraint(
    m_energy.J,
    rule=energy_power_hold_rule
)


# ============================================================
# OBJECTIVE — MIN ENERGY CONSUMPTION
# ============================================================

lambda_rlx = 1000.0


def min_energy_objective_rule(m):

    instantaneous_power = sum(
        m.P_max[j]
        * m.Y_active[j]
        for j in m.J
    )

    return (
        instantaneous_power
        + lambda_rlx * m.P_rlx
    )


m_energy.objective = pe.Objective(
    rule=min_energy_objective_rule,
    sense=pe.minimize
)

In [ ]:
# ============================================================
# MIN ENERGY RT LOOP
# ============================================================

from time import perf_counter

# Wall-clock time per control iteration; first iteration is included.
rt_iteration_time_energy_s = np.full(T, np.nan)
rt_solver_time_energy_s = np.full(T, np.nan)

solver_energy = pe.SolverFactory("highs")

variable_hysteresis = False


# ============================================================
# INITIAL STATE
# ============================================================

X_initial = X_onoff_FC_matrix[0, :].copy()

W_T_initial = initialize_time_weights_from_state(
    X0=X_initial,
    Type=Type,
    T_ON_0=T_ON_FC_matrix[0, :],
    T_OFF_0=T_OFF_FC_matrix[0, :],
    time_flex_available=time_flex_available_matrix[0, :],
)

count_initial = np.zeros(
    N_DEV,
    dtype=int
)


# ============================================================
# MINUTE-BY-MINUTE RT CONTROL
# ============================================================

for i in range(T):

    rt_iteration_started = perf_counter()

    # ========================================================
    # STEP 1 — CURRENT INPUTS
    # ========================================================

    P_lim = get_p_lim_for_minute(i)

    T_ON_now = T_ON_FC_matrix[i, :].copy()
    T_OFF_now = T_OFF_FC_matrix[i, :].copy()
    time_flex_available_now = time_flex_available_matrix[i, :]

    X_plan_vector = X_plan_ref_matrix[i, :].copy()
    Y_plan_vector = Y_plan_ref_matrix[i, :].copy()

    P_lim_energy_vector[i] = P_lim


    # ========================================================
    # STEP 2 — PREVIOUS RT STATE
    # ========================================================

    if i == 0:

        X_previous = X_initial.copy()

        Y_previous = np.ones(
            N_DEV,
            dtype=float
        )

        W_T_previous = W_T_initial.copy()

        count_previous = count_initial.copy()

    else:

        X_previous = X_energy_rt_matrix[
            i - 1, :
        ].copy()

        Y_previous = Y_energy_rt_matrix[
            i - 1, :
        ].copy()

        W_T_previous = W_T_energy_rt_matrix[
            i - 1, :
        ].copy()

        count_previous = count_hold_energy_matrix[
            i - 1, :
        ].copy()


    W_T_previous = prepare_time_weights_for_minute(
        prev_W_T=W_T_previous,
        X_reference=X_plan_vector,
        Type=Type,
        time_flex_available=time_flex_available_now,
        time_flex_start=time_flex_start_matrix[i, :],
    )

    # ========================================================
    # STEP 3 — TIME-FLEXIBLE PRE-PROCESSING
    # ========================================================

    if i == 0:

        X_before = X_plan_vector.copy()

        force_on_vector = np.zeros(
            N_DEV,
            dtype=int
        )

        force_off_vector = np.zeros(
            N_DEV,
            dtype=int
        )

    else:

        (
            X_before,
            force_on_vector,
            force_off_vector
        ) = function_dataset(

            i=i,

            X_RT_matrix=X_energy_rt_matrix,

            W_T_RT_matrix=W_T_energy_rt_matrix,

            X_plan_matrix=X_plan_ref_matrix,

            Type=Type,

            time_flex_available_matrix=time_flex_available_matrix,
            time_flex_start_matrix=time_flex_start_matrix,
            variable_hysteresis=variable_hysteresis
        )


    # ========================================================
    # STEP 4 — POWER-FLEXIBLE PRE-PROCESSING
    # ========================================================

    (
        Y_before,
        count_vector
    ) = update_power_before_optimization(

        X_before_opt_rt_vector=X_before,

        Y_plan_vector=Y_plan_vector,

        Y_prev_opt_rt_vector=Y_previous,

        count_prev_vector=count_previous,

        Type=Type
    )


    # ========================================================
    # STEP 5 — FLEXIBILITY WEIGHTS BEFORE OPTIMIZATION
    #
    # W_P is diagnostic only for Min Energy.
    # ========================================================

    W_P_before = compute_wp_vector(

        Y_vec=Y_before,

        Weight_P_min=Weight_P_min,

        Weight_P_max=Weight_P_max,

        Y_min=Y_min,

        Y_max=Y_max
    )


    # ========================================================
    # OPTIONAL — POWER BEFORE RT OPTIMIZATION
    # ========================================================

    P_before = compute_power_vector(

        X_vec=X_before,

        Power_max=Power_max,

        Y_vec=Y_before
    )

    P_before_energy_sum[i] = compute_power_sum(
        P_before
    )

    prevent_off_vector = np.zeros(
        N_DEV,
        dtype=int
    )

    for j in range(N_DEV):

        if (
            Type[j] == 2
            and time_flex_available_now[j]
            and X_before[j] == 1
            and force_on_vector[j] == 0
            and force_off_vector[j] == 0
        ):
            W_if_off = (
                W_T_previous[j]
                + 9.0 / T_OFF_now[j]
            )

            W_max_now = get_w_max(
                j,
                i,
                variable_hysteresis
            )

            prevent_off_vector[j] = int(
                W_if_off >= W_max_now - 1e-9
            )

    # ========================================================
    # STEP 6 — UPDATE OPTIMIZATION MODEL
    # ========================================================

    m_energy.P_lim.set_value(
        float(P_lim)
    )

    for j in range(N_DEV):

        m_energy.time_flex_available[j] = int(time_flex_available_now[j])

        m_energy.prevent_off[j] = int(prevent_off_vector[j])

        m_energy.X_in[j] = int(
            X_before[j]
        )

        m_energy.Y_in[j] = float(
            Y_before[j]
        )

        m_energy.force_on[j] = int(
            force_on_vector[j]
        )

        m_energy.force_off[j] = int(
            force_off_vector[j]
        )

        m_energy.count_T_Hold_Power[j] = int(
            count_vector[j]
        )

        m_energy.X_opt[j].value = int(
            X_before[j]
        )

        m_energy.Y_opt[j].value = float(
            Y_before[j]
        )


    m_energy.P_rlx.value = 0.0


    # ========================================================
    # STEP 7 — REBUILD DYNAMIC CONSTRAINTS
    # ========================================================

    if hasattr(
        m_energy,
        "status_logic"
    ):
        m_energy.del_component(
            m_energy.status_logic
        )

    if hasattr(
        m_energy,
        "power_hold_constraint"
    ):
        m_energy.del_component(
            m_energy.power_hold_constraint
        )


    m_energy.status_logic = pe.Constraint(
        m_energy.J,
        rule=energy_status_logic_rule
    )

    m_energy.power_hold_constraint = pe.Constraint(
        m_energy.J,
        rule=energy_power_hold_rule
    )


    # ========================================================
    # STEP 8 — MIN ENERGY OPTIMIZATION
    # ========================================================

    rt_solver_started = perf_counter()
    results = solver_energy.solve(
        m_energy,
        tee=False,
        load_solutions=False
    )

    rt_solver_time_energy_s[i] = perf_counter() - rt_solver_started

    termination = (
        results.solver.termination_condition
    )

    if (
        termination
        ==
        pe.TerminationCondition.optimal
    ):

        m_energy.solutions.load_from(
            results
        )

    else:

        raise RuntimeError(
            f"Min Energy optimization failed "
            f"at minute {i}: "
            f"{termination}"
        )


    # ========================================================
    # STEP 9 — EXECUTED RT DECISION
    # ========================================================

    X_final = np.zeros(
        N_DEV,
        dtype=int
    )

    Y_final = np.ones(
        N_DEV,
        dtype=float
    )

    for j in range(N_DEV):

        X_final[j] = int(
            round(
                pe.value(
                    m_energy.X_opt[j]
                )
            )
        )

        if X_final[j] == 1:

            Y_final[j] = float(
                pe.value(
                    m_energy.Y_opt[j]
                )
            )

        else:

            Y_final[j] = 1.0


    # ========================================================
    # STEP 10 — UPDATE POWER HOLD
    # ========================================================

    count_vector = (
        update_power_hold_after_optimization(

            X_opt_rt_vector=X_final,

            Y_before_opt_rt_vector=Y_before,

            Y_opt_rt_vector=Y_final,

            count_vector=count_vector,

            Type=Type,

            TimeHoldPower=TimeHoldPower
        )
    )


    # ========================================================
    # STEP 11 — FINAL RT STATES
    # ========================================================

    P_rlx_energy_vector[i] = float(
        pe.value(
            m_energy.P_rlx
        )
    )


    P_final = compute_power_vector(

        X_vec=X_final,

        Power_max=Power_max,

        Y_vec=Y_final
    )


    W_T_final = update_time_weights(

        prev_W_T=W_T_previous,

        X_now=X_final,

        Type=Type,

        T_ON_now=T_ON_now,

        T_OFF_now=T_OFF_now,

        minute=i,

        time_flex_available=time_flex_available_now,

        variable_hysteresis=variable_hysteresis
    )


    W_P_final = compute_wp_vector(

        Y_vec=Y_final,

        Weight_P_min=Weight_P_min,

        Weight_P_max=Weight_P_max,

        Y_min=Y_min,

        Y_max=Y_max
    )


    # ========================================================
    # STEP 12 — COMBINED WEIGHT VECTOR
    # ========================================================

    W_combined_final = np.full(
        N_DEV,
        np.nan,
        dtype=float
    )

    W_P_store = np.full(
        N_DEV,
        np.nan,
        dtype=float
    )

    for j in range(N_DEV):

        if Type[j] == 1:

            W_combined_final[j] = W_P_final[j]

            W_P_store[j] = W_P_final[j]

        elif Type[j] == 2:

            W_combined_final[j] = W_T_final[j]


    # ========================================================
    # STEP 13 — STORE RT STATE
    # ========================================================
    
    prevent_off_energy_rt_matrix[i, :] = prevent_off_vector

    X_energy_rt_matrix[
        i, :
    ] = X_final

    Y_energy_rt_matrix[
        i, :
    ] = Y_final

    W_T_energy_rt_matrix[
        i, :
    ] = W_T_final

    W_P_energy_rt_matrix[
        i, :
    ] = W_P_store

    W_energy_rt_matrix[
        i, :
    ] = W_combined_final

    P_energy_rt_matrix[
        i, :
    ] = P_final

    P_energy_rt_sum[i] = compute_power_sum(
        P_final
    )

    force_on_energy_rt_matrix[
        i, :
    ] = force_on_vector

    force_off_energy_rt_matrix[
        i, :
    ] = force_off_vector

    count_hold_energy_matrix[
        i, :
    ] = count_vector

    rt_iteration_time_energy_s[i] = perf_counter() - rt_iteration_started


In [ ]:
# ============================================================
# POTÊNCIA POR EQUIPAMENTO E POTÊNCIA AGREGADA [W]
# Executar depois de Max Utility e Min Energy
# ============================================================

import numpy as np

P_max = np.asarray(Power_max, dtype=float)[np.newaxis, :]

# Baseline
P_baseline_matrix = (
    np.asarray(X_onoff_FC_matrix)
    * np.asarray(Y_FC_matrix)
    * P_max
)
P_baseline_sum = P_baseline_matrix.sum(axis=1)

# Max Utility
P_utility_matrix = (
    np.asarray(X_utility_rt_matrix)
    * np.asarray(Y_utility_rt_matrix)
    * P_max
)
P_utility_sum = P_utility_matrix.sum(axis=1)

# Min Energy
P_energy_matrix = (
    np.asarray(X_energy_rt_matrix)
    * np.asarray(Y_energy_rt_matrix)
    * P_max
)
P_energy_sum = P_energy_matrix.sum(axis=1)

# Limite de potência por minuto [W]
P_lim_vector = np.array(
    [
        get_p_lim_for_minute(minute)
        for minute in range(len(P_baseline_sum))
    ],
    dtype=float,
)

# Real-time plots and metrics

## Daily Aggregate Consumption

In [ ]:
plot_stacked_device_power(
    X_matrix=X_onoff_FC_matrix,
    Y_matrix=Y_FC_matrix,
    Power_max=Power_max,
    dev_names=dev_names,

    time_start=0,
    time_finish=24*60,

    P_lim_vector=P_lim_utility_vector,

    colors=colors,

    tick_step=60,

    title="Baseline IK Consumption"
)
plot_stacked_device_power(
    X_matrix=X_utility_rt_matrix,
    Y_matrix=Y_utility_rt_matrix,
    Power_max=Power_max,
    dev_names=dev_names,

    time_start=0,
    time_finish=24*60,

    P_lim_vector=P_lim_utility_vector,

    colors=colors,

    tick_step=60,

    title="RT Control - Max Utility Objective"
)
plot_stacked_device_power(
    X_matrix=X_energy_rt_matrix,
    Y_matrix=Y_energy_rt_matrix,
    Power_max=Power_max,
    dev_names=dev_names,

    time_start=0,
    time_finish=24*60,

    P_lim_vector=P_lim_energy_vector,

    colors=colors,

    tick_step=60,

    title="RT Control - Min Energy Consumption Objective"
)

In [ ]:
power_limit_metrics = compute_power_limit_metrics(
    P_baseline=P_baseline_sum,
    P_utility=P_utility_sum,
    P_energy=P_energy_sum,
    P_lim=P_lim_vector,
    dt_minutes=1,
)

display(
    power_limit_metrics.style
    .hide(axis="index")
    .format(precision=2)
)
energy_metrics = compute_energy_metrics(
    P_baseline=P_baseline_sum,
    P_utility=P_utility_sum,
    P_energy=P_energy_sum,
    P_lim=P_lim_vector,
    time_zones=TIME_ZONES,
    tou_price_eur_kwh=TOU_PRICE_EUR_KWH,
    dt_minutes=1,
)

display(
    energy_metrics.style
    .hide(axis="index")
    .format(precision=2)
)
iteration_times = {
    "Max Utility": rt_iteration_time_utility_s,
    "Min Energy": rt_iteration_time_energy_s,
}
solver_times = {
    "Max Utility": rt_solver_time_utility_s,
    "Min Energy": rt_solver_time_energy_s,
}
control_period_s = 60.0

timing_rows = []
for scenario, values in iteration_times.items():
    total = np.asarray(values, dtype=float)
    solve = np.asarray(solver_times[scenario], dtype=float)

    if (
        total.ndim != 1
        or total.size == 0
        or total.shape != solve.shape
        or not np.all(np.isfinite(total))
        or not np.all(np.isfinite(solve))
    ):
        raise ValueError(f"Incomplete timing data for {scenario}.")

    timing_rows.append({
        "Scenario": scenario,
        "Iterations": total.size,
        "Mean iteration [ms]": total.mean() * 1000,
        "Median iteration [ms]": np.median(total) * 1000,
        "P95 iteration [ms]": np.percentile(total, 95) * 1000,
        "Max iteration [ms]": total.max() * 1000,
        "Mean solver call [ms]": solve.mean() * 1000,
        "Max / control period [%]": total.max() / control_period_s * 100,
        "Deadline misses": int(np.count_nonzero(total > control_period_s)),
    })

rt_computation_metrics = pd.DataFrame(timing_rows)
display(
    rt_computation_metrics.style
    .hide(axis="index")
    .format(precision=3)
)


In [ ]:
fig, ax = plot_time_flexible_comparison(
    device="Drinks Refrigerator",
    dev_names=dev_names,
    Type=Type,
    W_baseline=WT_FC_matrix,
    X_baseline=X_onoff_FC_matrix,
    W_utility=W_T_utility_rt_matrix,
    X_utility=X_utility_rt_matrix,
    W_energy=W_T_energy_rt_matrix,
    X_energy=X_energy_rt_matrix,
    time_start="00:00",
    time_finish="24:00",
    tick_step=60,
)

In [ ]:
fig, ax = plot_power_flexible_Y_comparison(
    device="Light2",  # Ou "Light2", "Exhaust", 10, 11, 12
    dev_names=dev_names,
    Type=Type,
    X_baseline=X_onoff_FC_matrix,
    Y_baseline=Y_FC_matrix,
    X_utility=X_utility_rt_matrix,
    Y_utility=Y_utility_rt_matrix,
    X_energy=X_energy_rt_matrix,
    Y_energy=Y_energy_rt_matrix,
    time_start=00 * 60,
    time_finish=24 * 60,
    tick_step=60,
)

In [ ]:
power_share_violation_summary, fig, ax = (
    plot_power_share_by_flexibility_at_violations(
        scenarios={
            "Baseline": {
                "X": X_onoff_FC_matrix,
                "Y": Y_FC_matrix,
                "P_lim": P_lim_utility_vector,
            },

            "Max Utility": {
                "X": X_utility_rt_matrix,
                "Y": Y_utility_rt_matrix,
                "P_lim": P_lim_utility_vector,
            },

            "Min Energy": {
                "X": X_energy_rt_matrix,
                "Y": Y_energy_rt_matrix,
                "P_lim": P_lim_energy_vector,
            },
        },

        Type=Type,
        Power_max=Power_max,

        # Analyse the complete 24-hour horizon.
        time_start=0,
        time_finish=24 * 60,
    )
)

## Load control behavior and appliance flexibility

In [ ]:
utility_metrics = compute_utility_metrics(
    X_forecast=X_onoff_FC_matrix,
    X_opt=X_utility_rt_matrix,
    Type=Type,
    dev_names=dev_names,
    dt_minutes=1,
)

display(
    utility_metrics.style
    .hide(axis="index")
    .format(precision=2)
)
utility_metrics = compute_utility_metrics(
    X_forecast=X_onoff_FC_matrix,
    X_opt=X_energy_rt_matrix,
    Type=Type,
    dev_names=dev_names,
    dt_minutes=1,
)

display(
    utility_metrics.style
    .hide(axis="index")
    .format(precision=2)
)

In [ ]:
# ============================================================
# BASELINE FLEXIBILITY FLAGS
# ============================================================
# Replay the forecast with the same temporal-weight evolution and switching
# rules used by RT. This gives the downward flexibility that would remain
# available if the forecast trajectory had been followed up to each minute.

X_baseline_rt_matrix = X_plan_ref_matrix.copy()
Y_baseline_rt_matrix = Y_plan_ref_matrix.copy()
W_T_baseline_rt_matrix = np.full((T, N_DEV), np.nan, dtype=float)
force_on_baseline_rt_matrix = np.zeros((T, N_DEV), dtype=int)
force_off_baseline_rt_matrix = np.zeros((T, N_DEV), dtype=int)
prevent_off_baseline_rt_matrix = np.zeros((T, N_DEV), dtype=int)
count_hold_baseline_matrix = np.zeros((T, N_DEV), dtype=int)
P_lim_baseline_vector = np.array(
    [get_p_lim_for_minute(minute) for minute in range(T)],
    dtype=float,
)

W_T_baseline_initial = initialize_time_weights_from_state(
    X0=X_baseline_rt_matrix[0, :],
    Type=Type,
    T_ON_0=T_ON_FC_matrix[0, :],
    T_OFF_0=T_OFF_FC_matrix[0, :],
    time_flex_available=time_flex_available_matrix[0, :],
)

baseline_rule_violations = []

for minute in range(T):
    if minute == 0:
        W_T_previous = W_T_baseline_initial.copy()
        X_before = X_plan_ref_matrix[minute, :].copy()
        force_on_vector = np.zeros(N_DEV, dtype=int)
        force_off_vector = np.zeros(N_DEV, dtype=int)
    else:
        W_T_previous = W_T_baseline_rt_matrix[minute - 1, :].copy()
        (
            X_before,
            force_on_vector,
            force_off_vector,
        ) = function_dataset(
            i=minute,
            X_RT_matrix=X_baseline_rt_matrix,
            W_T_RT_matrix=W_T_baseline_rt_matrix,
            X_plan_matrix=X_plan_ref_matrix,
            Type=Type,
            time_flex_available_matrix=time_flex_available_matrix,
            time_flex_start_matrix=time_flex_start_matrix,
            variable_hysteresis=False,
        )

    W_T_previous = prepare_time_weights_for_minute(
        prev_W_T=W_T_previous,
        X_reference=X_plan_ref_matrix[minute, :],
        Type=Type,
        time_flex_available=time_flex_available_matrix[minute, :],
        time_flex_start=time_flex_start_matrix[minute, :],
    )

    prevent_off_vector = np.zeros(N_DEV, dtype=int)
    for j in range(N_DEV):
        if (
            Type[j] == 2
            and time_flex_available_matrix[minute, j]
            and X_before[j] == 1
            and force_on_vector[j] == 0
            and force_off_vector[j] == 0
        ):
            W_if_off = W_T_previous[j] + 9.0 / T_OFF_FC_matrix[minute, j]
            prevent_off_vector[j] = int(
                W_if_off >= get_w_max(j, minute, False) - 1e-9
            )

    X_forecast_now = X_baseline_rt_matrix[minute, :]
    invalid = (
        ((force_on_vector == 1) & (X_forecast_now != 1))
        | ((force_off_vector == 1) & (X_forecast_now != 0))
    )
    for j in np.flatnonzero(invalid):
        baseline_rule_violations.append({
            "Minute": minute,
            "Time": f"{minute // 60:02d}:{minute % 60:02d}",
            "Device": dev_names[j],
            "Forecast state": int(X_forecast_now[j]),
            "Force ON": int(force_on_vector[j]),
            "Force OFF": int(force_off_vector[j]),
        })

    W_T_baseline_rt_matrix[minute, :] = update_time_weights(
        prev_W_T=W_T_previous,
        X_now=X_forecast_now,
        Type=Type,
        T_ON_now=T_ON_FC_matrix[minute, :],
        T_OFF_now=T_OFF_FC_matrix[minute, :],
        minute=minute,
        time_flex_available=time_flex_available_matrix[minute, :],
        variable_hysteresis=False,
    )
    force_on_baseline_rt_matrix[minute, :] = force_on_vector
    force_off_baseline_rt_matrix[minute, :] = force_off_vector
    prevent_off_baseline_rt_matrix[minute, :] = prevent_off_vector

baseline_rule_violations = pd.DataFrame(baseline_rule_violations)
if not baseline_rule_violations.empty:
    raise RuntimeError(
        "The forecast violates the RT force_on/force_off rules; inspect "
        "baseline_rule_violations before interpreting baseline flexibility."
    )

if not np.allclose(
    W_T_baseline_rt_matrix[:, Type == 2],
    WT_FC_matrix[:, Type == 2],
    rtol=0,
    atol=1e-10,
    equal_nan=True,
):
    raise RuntimeError(
        "The replayed baseline weights do not match WT_FC_matrix."
    )

(
    metrics_baseline,
    violations_baseline,
    device_summary_baseline,
) = analyze_rt_flexibility(
    X_reference=X_plan_ref_matrix,
    Y_reference=Y_plan_ref_matrix,
    X_rt=X_baseline_rt_matrix,
    Y_rt=Y_baseline_rt_matrix,
    force_on=force_on_baseline_rt_matrix,
    force_off=force_off_baseline_rt_matrix,
    prevent_off=prevent_off_baseline_rt_matrix,
    count_hold_after=count_hold_baseline_matrix,
    Type=Type,
    Power_max=Power_max,
    Y_min=Y_min,
    dev_names=dev_names,
    P_lim=P_lim_baseline_vector,
)

(
    metrics_utility,
    violations_utility,
    device_summary_utility,
) = analyze_rt_flexibility(
    X_reference=X_plan_ref_matrix,
    Y_reference=Y_plan_ref_matrix,
    X_rt=X_utility_rt_matrix,
    Y_rt=Y_utility_rt_matrix,
    force_on=force_on_utility_rt_matrix,
    force_off=force_off_utility_rt_matrix,
    prevent_off=prevent_off_utility_rt_matrix,
    count_hold_after=count_hold_utility_matrix,
    Type=Type,
    Power_max=Power_max,
    Y_min=Y_min,
    dev_names=dev_names,
    P_lim=P_lim_utility_vector,
)


(
    metrics_energy,
    violations_energy,
    device_summary_energy,
) = analyze_rt_flexibility(
    X_reference=X_plan_ref_matrix,
    Y_reference=Y_plan_ref_matrix,
    X_rt=X_energy_rt_matrix,
    Y_rt=Y_energy_rt_matrix,
    force_on=force_on_energy_rt_matrix,
    force_off=force_off_energy_rt_matrix,
    prevent_off=prevent_off_energy_rt_matrix,
    count_hold_after=count_hold_energy_matrix,
    Type=Type,
    Power_max=Power_max,
    Y_min=Y_min,
    dev_names=dev_names,
    P_lim=P_lim_energy_vector,
)

In [ ]:
fig, axes = plot_rt_flexibility(
    {
        "Max Utility": metrics_utility,
        "Min Energy": metrics_energy,
    },
    time_start=0 * 60,
    time_finish=24 * 60,
    tick_step=120,
)

plt.show()

In [ ]:
flexibility_summary = summarize_rt_flexibility(
    {
        "Max Utility": metrics_utility,
        "Min Energy": metrics_energy,
    },
    time_start=0 * 60,
    time_finish=24 * 60
)

display(
    flexibility_summary.style
    .hide(axis="index")
    .format({
        "Average available flexibility [kW]": "{:.2f}",
        "Minutes requiring reduction": "{:.0f}",
        "Minutes with sufficient flexibility": "{:.0f}",
        "Sufficient-flexibility rate [%]": "{:.1f}",
    })
)

## Reconstructing Fridge Temperature

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# ============================================================
# Fixed thermal-model configuration
# ============================================================

FRIDGE_INDEX = 0  # Drinks Refrigerator

P_THERMAL = 60.0       # W
A_THERMAL = 3.21       # W/K
ETA_THERMAL = 3.0      # dimensionless
T_AMBIENT = 18.0       # °C
T_INITIAL = 4.0        # °C
H_MINUS = 2.0          # °C
H_PLUS = 6.0           # °C
DT_MINUTES = 1

MC_MIN = 8.0           # Wh/K
MC_MAX = 30.0          # Wh/K
MC_STEP = 0.01         # Wh/K


# Baseline activations of the Drinks Refrigerator
X_base = np.asarray(
    X_onoff_FC_matrix[:, FRIDGE_INDEX],
    dtype=int
)


# ============================================================
# Temperature reconstruction
# ============================================================

def reconstruct_fridge_temperature(
    X,
    mc,
    T0=T_INITIAL,
    P=P_THERMAL,
    A=A_THERMAL,
    eta=ETA_THERMAL,
    Tamb=T_AMBIENT,
    dt_minutes=DT_MINUTES
):
    """
    Reconstruct the refrigerator temperature from a binary
    compressor activation sequence.

    Parameters
    ----------
    X : array-like
        Compressor activation state: 0 = OFF, 1 = ON.
    mc : float
        Equivalent thermal mass [Wh/K].
    """

    X = np.asarray(X, dtype=int)

    if X.ndim != 1:
        raise ValueError("X must be a one-dimensional activation sequence.")

    if not np.all(np.isin(X, [0, 1])):
        raise ValueError("X must contain only binary values 0 and 1.")

    T = np.zeros(len(X), dtype=float)
    T[0] = T0

    dt_hours = dt_minutes / 60.0
    epsilon = np.exp(-(dt_hours * A) / mc)

    for i in range(len(X) - 1):
        compressor_power = P * X[i]

        T[i + 1] = (
            epsilon * (T[i] - Tamb)
            + Tamb
            - eta * (compressor_power / A) * (1.0 - epsilon)
        )

    return T


# ============================================================
# Select the effective thermal mass
# ============================================================

def select_effective_thermal_mass(
    X_base,
    mc_min=MC_MIN,
    mc_max=MC_MAX,
    mc_step=MC_STEP,
    H_minus=H_MINUS,
    H_plus=H_PLUS,
    temperature_tolerance=1e-9
):
    """
    Select an effective thermal mass from the literature range.

    Primary rule:
        Among the candidates that keep the complete baseline
        trajectory inside the thermal interval, select the value
        with the lowest switching-temperature RMSE.

    Fallback rule:
        If no candidate remains completely inside the interval,
        minimize a score combining switching error and band
        violation.

    OFF -> ON transitions are expected near H_plus.
    ON  -> OFF transitions are expected near H_minus.
    """

    X_base = np.asarray(X_base, dtype=int)

    # Index i represents the first sample in the new state.
    off_to_on_idx = np.flatnonzero(
        (X_base[:-1] == 0) & (X_base[1:] == 1)
    ) + 1

    on_to_off_idx = np.flatnonzero(
        (X_base[:-1] == 1) & (X_base[1:] == 0)
    ) + 1

    if len(off_to_on_idx) == 0:
        raise ValueError("No OFF-to-ON transitions were found.")

    if len(on_to_off_idx) == 0:
        raise ValueError("No ON-to-OFF transitions were found.")

    mc_values = np.arange(
        mc_min,
        mc_max + mc_step / 2.0,
        mc_step
    )

    records = []
    trajectories = {}

    for mc in mc_values:
        T = reconstruct_fridge_temperature(
            X=X_base,
            mc=mc
        )

        upper_switch_errors = T[off_to_on_idx] - H_plus
        lower_switch_errors = T[on_to_off_idx] - H_minus

        all_switch_errors = np.concatenate([
            upper_switch_errors,
            lower_switch_errors
        ])

        switch_rmse = np.sqrt(
            np.mean(all_switch_errors ** 2)
        )

        upper_violation = np.maximum(T - H_plus, 0.0)
        lower_violation = np.maximum(H_minus - T, 0.0)

        outside_mask = (
            (T > H_plus + temperature_tolerance)
            | (T < H_minus - temperature_tolerance)
        )

        band_violation_rmse = np.sqrt(
            np.mean(
                upper_violation ** 2
                + lower_violation ** 2
            )
        )

        max_upper_violation = np.max(upper_violation)
        max_lower_violation = np.max(lower_violation)

        records.append({
            "mc [Wh/K]": float(mc),
            "Switch RMSE [°C]": float(switch_rmse),
            "OFF-ON RMSE [°C]": float(
                np.sqrt(np.mean(upper_switch_errors ** 2))
            ),
            "ON-OFF RMSE [°C]": float(
                np.sqrt(np.mean(lower_switch_errors ** 2))
            ),
            "Minutes outside interval": int(
                np.sum(outside_mask)
            ),
            "Time inside interval [%]": float(
                100.0 * np.mean(~outside_mask)
            ),
            "Band violation RMSE [°C]": float(
                band_violation_rmse
            ),
            "Maximum upper violation [°C]": float(
                max_upper_violation
            ),
            "Maximum lower violation [°C]": float(
                max_lower_violation
            ),
            "Minimum temperature [°C]": float(np.min(T)),
            "Maximum temperature [°C]": float(np.max(T))
        })

        trajectories[round(float(mc), 8)] = T

    diagnostics = pd.DataFrame(records)

    # First try to find candidates entirely inside the band.
    feasible = diagnostics[
        diagnostics["Minutes outside interval"] == 0
    ].copy()

    if not feasible.empty:
        best_row = feasible.sort_values(
            by=[
                "Switch RMSE [°C]",
                "mc [Wh/K]"
            ]
        ).iloc[0]

        selection_rule = (
            "Lowest switching RMSE among candidates "
            "entirely inside the thermal interval."
        )

    else:
        # Fallback if no mass in the literature range is fully feasible.
        # The band violation is strongly penalized, while switching
        # accuracy is still considered.
        diagnostics["Fallback score"] = (
            diagnostics["Switch RMSE [°C]"]
            + 10.0 * diagnostics["Band violation RMSE [°C]"]
        )

        best_row = diagnostics.sort_values(
            by=[
                "Fallback score",
                "Switch RMSE [°C]",
                "mc [Wh/K]"
            ]
        ).iloc[0]

        selection_rule = (
            "No candidate remained entirely inside the interval; "
            "the value with the lowest fallback score was selected."
        )

    best_mc = float(best_row["mc [Wh/K]"])
    best_temperature = trajectories[round(best_mc, 8)]

    return {
        "mc": best_mc,
        "temperature": best_temperature,
        "diagnostics": diagnostics,
        "best_metrics": best_row,
        "selection_rule": selection_rule,
        "off_to_on_indices": off_to_on_idx,
        "on_to_off_indices": on_to_off_idx
    }


# ============================================================
# Run the parameter selection
# ============================================================

thermal_mass_selection = select_effective_thermal_mass(
    X_base=X_base
)

selected_mc = thermal_mass_selection["mc"]
T_baseline = thermal_mass_selection["temperature"]
mc_diagnostics = thermal_mass_selection["diagnostics"]

print(f"Selected thermal mass: {selected_mc:.2f} Wh/K")
print(thermal_mass_selection["selection_rule"])

display(
    thermal_mass_selection["best_metrics"]
    .to_frame(name="Value")
)

In [ ]:
T_max_utility = reconstruct_fridge_temperature(
    X=X_utility_rt_matrix[:, FRIDGE_INDEX],
    mc=selected_mc
)

T_min_energy = reconstruct_fridge_temperature(
    X=X_energy_rt_matrix[:, FRIDGE_INDEX],
    mc=selected_mc
)

In [ ]:
plot_fridge_temperature_smartfridge(
    T_baseline=T_baseline,
    T_opt=T_max_utility,
    X_baseline=X_base,
    X_opt=X_utility_rt_matrix[:, FRIDGE_INDEX],
    SP=4.0,
    H_minus=2.0,
    H_plus=6.0,
    tick_step=60,
)

plot_fridge_temperature_smartfridge(
    T_baseline=T_baseline,
    T_opt=T_min_energy,
    X_baseline=X_base,
    X_opt=X_energy_rt_matrix[:, FRIDGE_INDEX],
    SP=4.0,
    H_minus=2.0,
    H_plus=6.0,
    tick_step=60,
)

In [ ]:
fridge_temperature_metrics = compute_fridge_temperature_metrics(
    T_baseline=T_baseline,
    T_utility=T_max_utility,
    T_energy=T_min_energy,
    SP=4.0,
    H_minus=2.0,
    H_plus=6.0,
    dt_minutes=1
)

display(fridge_temperature_metrics)

# Quantification of Flexibility

This section presents three complementary views of the flexibility represented by the operational model.

1. **Available downward flexibility along an operating trajectory.** The baseline, Max Utility, and Min Energy scenarios are compared using the same pre-decision diagnostic. This measures the instantaneous reduction available under the state and weight history produced by each trajectory.
2. **Pointwise aggregate flexibility envelope.** Following the set-based interpretation of aggregate flexibility, each time-flexible appliance is represented by its set of complete admissible daily power profiles. The aggregate set is the sum of the individual sets. The plotted lower and upper boundaries are the pointwise projections of that set.
3. **Two complete priority trajectories.** An OFF-priority trajectory selects OFF whenever the existing RT rules permit it; an ON-priority trajectory selects ON whenever permitted. These trajectories illustrate temporal coupling. They are admissible members of the aggregate set, but they are not its lower and upper boundaries and may cross.

The envelope calculation uses the existing forecast initial conditions, `T_ON`, `T_OFF`, availability windows, weight recurrence, `force_on`, `force_off`, and `prevent_off`. No kitchen-level power constraint or Demand Response request is imposed. It also preserves the existing RT interpretation of the limits: the switching rules react to the previous weight and the weight is updated without clipping. No thermal or virtual-battery model is introduced.

## Downward Flexibility Along the Baseline and RT Trajectories

For each trajectory, the available reduction is evaluated immediately before the current optimization decision. Time-flexible appliances contribute when they are ON and can be switched OFF; power-flexible appliances contribute the reducible power above `Y_min` when no power-hold constraint is active. The dashed curve is the reduction required to meet the kitchen power limit. For the baseline, red markers represent power-limit exceedances in the uncontrolled reference; for the RT strategies they represent violations remaining after control.

The figure retains the 17:00–19:00 interval used in the RT results. The table reports the same metrics over the complete day.

In [ ]:
quantification_downward_scenarios = {
    "Baseline": metrics_baseline,
    "Max Utility": metrics_utility,
    "Min Energy": metrics_energy,
}

downward_time_start = 17 * 60
downward_time_finish = 19 * 60
downward_tick_step = 15

fig_downward, axes_downward = plot_rt_flexibility(
    quantification_downward_scenarios,
    time_start=downward_time_start,
    time_finish=downward_time_finish,
    tick_step=downward_tick_step,
)
plt.show()

summary_time_start = 17 * 60
summary_time_finish = 19 * 60

quantification_downward_summary = summarize_rt_flexibility(
    quantification_downward_scenarios,
    time_start=summary_time_start,
    time_finish=summary_time_finish,
)
display(
    quantification_downward_summary.style
    .hide(axis="index")
    .format({
        "Average available flexibility [kW]": "{:.2f}",
        "Minutes requiring reduction": "{:.0f}",
        "Minutes with sufficient flexibility": "{:.0f}",
        "Sufficient flexibility rate [%]": "{:.1f}",
    })
)


## Pointwise Aggregate Flexibility Envelope

Let $\mathcal{X}_j$ be the set of complete admissible binary trajectories for time-flexible appliance $j$. Its power-profile set is

$$\mathcal{P}_j=\left\{\mathbf p_j:\ p_{j,t}=P_{\max,j}x_{j,t},\ \mathbf x_j\in\mathcal X_j\right\}.$$

Because there are no shared constraints in this characterization, the aggregate profile set is the Minkowski sum

$$\mathcal P_{\mathrm{agg}}=\bigoplus_{j:\,\mathrm{Type}_j=2}\mathcal P_j.$$

The plotted boundaries are

$$P_{\mathrm{agg,min}}(t)=\min_{\mathbf p\in\mathcal P_{\mathrm{agg}}}p_t,$$

$$P_{\mathrm{agg,max}}(t)=\max_{\mathbf p\in\mathcal P_{\mathrm{agg}}}p_t.$$

Each endpoint is supported by a complete feasible daily trajectory and is replayed through the existing RT rules. Different endpoints can require different trajectories. Consequently, the complete lower or upper boundary curve need not itself be an admissible daily trajectory, and the shaded band is an instantaneous projection rather than a complete representation of temporal feasibility.

`Power_max` remains in W throughout the device calculation and is divided by 1000 only when the aggregate results are reported in kW.

In [ ]:
from bisect import bisect_right
from time import perf_counter
import struct

# All powers below stay in W until the final table/plot conversion to kW.
# Closed intervals contain representable float64 weights. Inverse transitions
# preserve the notebook's floating-point comparisons, including equality cases.
def _fq_key(x):
    bits = struct.unpack('>Q', struct.pack('>d', float(x)))[0]
    return (~bits & ((1 << 64) - 1)) if bits >> 63 else bits ^ (1 << 63)


def _fq_float(key):
    bits = key ^ (1 << 63) if key >> 63 else ~key & ((1 << 64) - 1)
    return struct.unpack('>d', struct.pack('>Q', bits))[0]


def _fq_first_ge(value, delta, lower, upper, strict=False):
    """First representable w in [lower, upper] with fl(w+delta) >= value.

    strict=True uses >. None means no such weight. Usually the inverse is
    within a few ulps of value-delta; integer bisection handles cancellation.
    """
    def meets(w):
        y = w + delta
        return y > value if strict else y >= value

    if meets(lower):
        return lower
    if not meets(upper):
        return None
    candidate = min(upper, max(lower, value - delta))
    for _ in range(4):
        if meets(candidate):
            previous = float(np.nextafter(candidate, -np.inf))
            if previous < lower or not meets(previous):
                return candidate
            candidate = previous
        else:
            candidate = float(np.nextafter(candidate, np.inf))
    left, right = _fq_key(lower), _fq_key(upper)
    while right - left > 1:
        middle = (left + right) // 2
        if meets(_fq_float(middle)):
            right = middle
        else:
            left = middle
    return _fq_float(right)


def _fq_preimage(interval, delta, domain):
    """Exact float64 inverse of one translated interval, within action domain."""
    lo, hi = interval
    lower, upper = domain
    if lower > upper:
        return None
    first = _fq_first_ge(lo, delta, lower, upper)
    if first is None:
        return None
    after = _fq_first_ge(hi, delta, lower, upper, strict=True)
    last = upper if after is None else float(np.nextafter(after, -np.inf))
    return (first, last) if first <= last else None


def _fq_union(intervals):
    """Merge overlapping/adjacent floating intervals, never fill a gap."""
    merged = []
    for lo, hi in sorted(intervals):
        if merged and lo <= np.nextafter(merged[-1][1], np.inf):
            merged[-1] = (merged[-1][0], max(hi, merged[-1][1]))
        else:
            merged.append((lo, hi))
    return merged


def _fq_contains(intervals, w):
    k = bisect_right(intervals, (float(w), np.inf)) - 1
    return k >= 0 and intervals[k][0] <= w <= intervals[k][1]


def _fq_device_setup(j):
    available = np.asarray(time_flex_available_matrix[:, j], dtype=bool)
    active = np.flatnonzero(available)
    if not active.size:
        return None
    start, stop = int(active[0]), int(active[-1]) + 1
    if not np.all(available[start:stop]):
        raise ValueError('This characterization requires one contiguous window per device.')
    reference = np.asarray(X_onoff_FC_matrix[start:stop, j], dtype=int)
    ton = np.asarray(T_ON_FC_matrix[start:stop, j], dtype=float)
    toff = np.asarray(T_OFF_FC_matrix[start:stop, j], dtype=float)
    if not (np.all(np.isfinite(ton)) and np.all(np.isfinite(toff))
            and np.all(ton > 0) and np.all(toff > 0)):
        raise ValueError(f'Invalid active durations for {dev_names[j]}.')
    delta = np.column_stack((9.0 / toff, -9.0 / ton))
    initial = 10.0 if reference[0] else 1.0
    # Outward-rounded bound containing every possible prefix, even before
    # applying switching rules. It is a search bound, not a new constraint.
    lower = upper = initial
    for off_delta, on_delta in delta:
        lower = float(np.nextafter(lower + on_delta, -np.inf))
        upper = float(np.nextafter(upper + off_delta, np.inf))
    domains = []
    for k, (off_delta, _) in enumerate(delta):
        minute = start + k
        ceiling = float(get_w_max(j, minute, False))
        entry = minute == 0 or bool(time_flex_start_matrix[minute, j])
        if reference[k]:
            first_blocked = _fq_first_ge(ceiling - 1e-9, float(off_delta), lower, upper)
            off_limit = upper if first_blocked is None else float(np.nextafter(first_blocked, -np.inf))
        else:
            off_limit = upper
        if entry:
            domain_off = (lower, min(upper, off_limit))
            domain_on = (lower, upper)
        else:
            # w <= 1 forces OFF and bypasses prevent_off. w >= ceiling
            # forces ON. In the interior, prevent_off depends on reference X.
            domain_off = (lower, min(upper, max(1.0, min(
                float(np.nextafter(ceiling, -np.inf)), off_limit))))
            domain_on = (max(lower, float(np.nextafter(1.0, np.inf))), upper)
        domains.append((domain_off, domain_on))
    return dict(j=j, start=start, stop=stop, reference=reference,
                delta=delta, initial=initial, domains=domains)


def _fq_allowed(problem, k, w, action):
    lo, hi = problem['domains'][k][action]
    return lo <= w <= hi


def _fq_path(problem, preference=0, target=None):
    """Find a full-window witness, or prove a target (minute, action) infeasible.

    Backward sets describe weights from which the requested action is reachable.
    With no terminal condition every allowed action admits a continuation:
    ordinary minutes force OFF for w<=1 and permit ON for w>1; entry permits ON.
    """
    n = len(problem['reference'])
    preferences = np.broadcast_to(np.asarray(preference, dtype=int), (n,))
    backward = None
    if target is not None:
        t, action = target
        domain = problem['domains'][t][action]
        if domain[0] > domain[1]:
            return None, 0
        backward = [None] * (t + 1)
        backward[t] = [domain]
        peak_intervals = 1
        for k in range(t - 1, -1, -1):
            intervals = []
            for a in (0, 1):
                for interval in backward[k + 1]:
                    preimage = _fq_preimage(interval, float(problem['delta'][k, a]),
                                            problem['domains'][k][a])
                    if preimage is not None:
                        intervals.append(preimage)
            backward[k] = _fq_union(intervals)
            peak_intervals = max(peak_intervals, len(backward[k]))
            if not backward[k]:
                return None, peak_intervals
        if not _fq_contains(backward[0], problem['initial']):
            return None, peak_intervals
    else:
        peak_intervals = 0
    w = problem['initial']
    path = np.empty(n, dtype=np.int8)
    for k in range(n):
        preferred = int(preferences[k])
        choices = (target[1],) if target is not None and k == target[0] else (preferred, 1 - preferred)
        for a in choices:
            next_w = w + float(problem['delta'][k, a])
            if _fq_allowed(problem, k, w, a) and (
                backward is None or k >= target[0] or _fq_contains(backward[k + 1], next_w)
            ):
                path[k], w = a, next_w
                break
        else:
            raise AssertionError('Backward reachability witness reconstruction failed.')
    return path, peak_intervals


def fq_replay(j, trajectory, compare_forecast_weights=False):
    """Independent full-day replay using the actual RT helper functions.

    Records every violation without repairing the supplied trajectory.
    No kitchen power limit, objective, or DR request is involved.
    """
    trajectory = np.asarray(trajectory)
    if trajectory.shape != (T,) or not np.all(np.isin(trajectory, (0, 1))):
        raise ValueError('A witness must be a complete binary daily trajectory.')
    # A single-device view keeps replay inexpensive while retaining notebook helpers.
    local_type = np.array([2])
    reference = X_onoff_FC_matrix[:, j:j+1]
    available = time_flex_available_matrix[:, j:j+1]
    starts = time_flex_start_matrix[:, j:j+1]
    ton = T_ON_FC_matrix[:, j:j+1]
    toff = T_OFF_FC_matrix[:, j:j+1]
    weights = np.full((T, 1), np.nan)
    states = trajectory[:, None]
    previous = initialize_time_weights_from_state(reference[0], local_type, ton[0], toff[0], available[0])
    failures = []
    for minute in range(T):
        previous = prepare_time_weights_for_minute(previous, reference[minute], local_type,
                                                  available[minute], starts[minute])
        before, force_on, force_off = function_dataset(
            minute, states, weights, reference, local_type, available, starts,
            variable_hysteresis=False)
        reasons = []
        x = int(trajectory[minute])
        if not available[minute, 0]:
            if x != reference[minute, 0]:
                reasons.append('outside-window reference')
        else:
            if force_on[0] and x != 1:
                reasons.append('force_on')
            if force_off[0] and x != 0:
                reasons.append('force_off')
            if (before[0] == 1 and not force_on[0] and not force_off[0]
                    and previous[0] + 9.0 / toff[minute, 0] >= get_w_max(j, minute, False) - 1e-9
                    and x != 1):
                reasons.append('prevent_off')
        if reasons:
            failures.append(dict(device=dev_names[j], minute=minute,
                                 time=f'{minute // 60:02d}:{minute % 60:02d}',
                                 state=x, previous_weight=previous[0], rule=', '.join(reasons)))
        previous = update_time_weights(previous, np.array([x]), local_type,
                                       ton[minute], toff[minute], minute, available[minute], False)
        weights[minute] = previous
    weight_match = True
    if compare_forecast_weights:
        weight_match = bool(np.allclose(weights[:, 0], WT_FC_matrix[:, j],
                                        rtol=0, atol=1e-10, equal_nan=True))
    return failures, weight_match


def fq_device_envelope(j):
    started = perf_counter()
    problem = _fq_device_setup(j)
    reference = np.asarray(X_onoff_FC_matrix[:, j], dtype=np.int8)
    witnesses = []
    certificates = np.full((T, 2), -1, dtype=int)
    queries = infeasible = peak = 0

    def register(path):
        # Validate each complete witness using the original RT functions.
        failures, _ = fq_replay(j, path)
        if failures:
            raise AssertionError(f'Invalid endpoint witness: {failures[0]}')
        index = len(witnesses)
        witnesses.append(path.copy())
        certificates[np.arange(T), path] = index

    if problem is None:
        register(reference)
    else:
        for preference in (0, 1):
            local, _ = _fq_path(problem, preference)
            full = reference.copy()
            full[problem['start']:problem['stop']] = local
            register(full)
        for t in range(problem['start'], problem['stop']):
            for action in (0, 1):
                if certificates[t, action] >= 0:
                    continue
                queries += 1
                # Prefer still-uncovered actions along the witness to certify many
                # other minutes in the same traversal. This changes search order only.
                coverage = certificates[problem['start']:problem['stop']]
                preference = np.where(coverage[:, 0] < 0, 0,
                                      np.where(coverage[:, 1] < 0, 1, problem['reference']))
                local, n_intervals = _fq_path(problem, preference, (t - problem['start'], action))
                peak = max(peak, n_intervals)
                if local is None:
                    infeasible += 1
                else:
                    full = reference.copy()
                    full[problem['start']:problem['stop']] = local
                    register(full)
    possible = certificates >= 0
    if not np.all(possible.any(axis=1)):
        raise AssertionError('Empty device flexibility set.')
    xmin = (~possible[:, 0]).astype(int)
    xmax = possible[:, 1].astype(int)
    for t in range(T):
        for a in (xmin[t], xmax[t]):
            assert witnesses[certificates[t, a]][t] == a
    return dict(xmin=xmin, xmax=xmax, witnesses=np.array(witnesses),
                certificates=certificates, device=dev_names[j],
                reachability_queries=queries, infeasible_queries=infeasible,
                max_interval_count=peak, validated_witnesses=len(witnesses),
                seconds=perf_counter() - started)


In [ ]:
fq_devices = np.flatnonzero(np.asarray(Type) == 2)
if T != 1440 or not len(fq_devices):
    raise ValueError("Expected a 1440-minute horizon with at least one Type 2 appliance.")

# Verify that the forecast itself is an admissible member of the same set.
fq_forecast_validation_rows = []
for fq_j in fq_devices:
    fq_failures, fq_weight_match = fq_replay(
        fq_j,
        X_onoff_FC_matrix[:, fq_j],
        compare_forecast_weights=True,
    )
    fq_forecast_validation_rows.append({
        "Device": dev_names[fq_j],
        "Rule violations [min]": len(fq_failures),
        "Forecast weights match": fq_weight_match,
        "Feasible": not fq_failures and fq_weight_match,
    })

fq_forecast_validation = pd.DataFrame(fq_forecast_validation_rows)
if not fq_forecast_validation["Feasible"].all():
    display(fq_forecast_validation)
    raise RuntimeError("The forecast is not feasible under the envelope rules.")

# Per-device feasible sets can be characterized independently and then added.
fq_results = {int(j): fq_device_envelope(int(j)) for j in fq_devices}
fq_xmin = np.column_stack([fq_results[int(j)]["xmin"] for j in fq_devices])
fq_xmax = np.column_stack([fq_results[int(j)]["xmax"] for j in fq_devices])
fq_xforecast = X_onoff_FC_matrix[:, fq_devices]
fq_power_W = np.asarray(Power_max, dtype=float)[fq_devices]

P_agg_min = fq_xmin @ fq_power_W / 1000.0
P_agg_max = fq_xmax @ fq_power_W / 1000.0
P_agg_FC = fq_xforecast @ fq_power_W / 1000.0
N_agg_min = fq_xmin.sum(axis=1)
N_agg_max = fq_xmax.sum(axis=1)
N_agg_FC = fq_xforecast.sum(axis=1)

if not np.all((P_agg_min <= P_agg_FC + 1e-10) &
              (P_agg_FC <= P_agg_max + 1e-10)):
    raise AssertionError("The feasible forecast must lie inside the aggregate envelope.")

F_agg_down = P_agg_FC - P_agg_min
F_agg_up = P_agg_max - P_agg_FC
aggregate_flexibility_indicators = pd.DataFrame({
    "Indicator": [
        "Maximum downward flexibility",
        "Maximum upward flexibility",
        "Mean downward flexibility",
        "Mean upward flexibility",
    ],
    "Value [kW]": [
        F_agg_down.max(), F_agg_up.max(),
        F_agg_down.mean(), F_agg_up.mean(),
    ],
})
display(
    aggregate_flexibility_indicators.style
    .hide(axis="index")
    .format({"Value [kW]": "{:.3f}"})
)

fq_validation_summary = pd.DataFrame([
    {
        "Device": result["device"],
        "Reachability queries": result["reachability_queries"],
        "Infeasible endpoint queries": result["infeasible_queries"],
        "Validated full-day witnesses": result["validated_witnesses"],
        "Maximum interval count": result["max_interval_count"],
        "Computation time [s]": result["seconds"],
    }
    for result in fq_results.values()
])
display(fq_validation_summary.style.hide(axis="index").format({
    "Computation time [s]": "{:.3f}",
}))
print("All pointwise endpoints are certified by complete feasible device trajectories.")


In [ ]:
def _fq_plot_window(time_start, time_finish, tick_step):
    if not (0 <= time_start < time_finish <= T):
        raise ValueError("Expected 0 <= time_start < time_finish <= T.")
    if not isinstance(tick_step, (int, np.integer)) or tick_step <= 0:
        raise ValueError("tick_step must be a positive integer number of minutes.")

    time_start = int(time_start)
    time_finish = int(time_finish)
    minutes = np.arange(time_start, time_finish + 1)
    hours = minutes / 60.0

    ticks = np.arange(time_start, time_finish + 1, tick_step, dtype=int)
    if ticks[-1] != time_finish:
        ticks = np.r_[ticks, time_finish]
    tick_hours = ticks / 60.0
    tick_labels = [f"{minute // 60:02d}:{minute % 60:02d}" for minute in ticks]

    def step_window(values):
        values = np.asarray(values)
        selected = values[time_start:time_finish]
        return np.r_[selected, selected[-1]]

    return hours, tick_hours, tick_labels, step_window


def plot_aggregate_flexibility_envelope(
    time_start=0,
    time_finish=24 * 60,
    tick_step=120,
):
    """Plot a selected interval of the pointwise aggregate envelope."""
    hours, tick_hours, tick_labels, step_window = _fq_plot_window(
        time_start, time_finish, tick_step
    )

    with plt.rc_context({
        "font.size": 10,
        "axes.titlesize": 11,
        "axes.labelsize": 10,
        "legend.fontsize": 9,
        "axes.spines.top": False,
        "axes.spines.right": False,
    }):
        fig, axes = plt.subplots(
            2, 1, figsize=(12, 6.8), sharex=True,
            gridspec_kw={"height_ratios": [1.5, 1]}, constrained_layout=True,
        )
        axes[0].fill_between(
            hours, step_window(P_agg_min), step_window(P_agg_max), step="post",
            color="#56B4E9", alpha=0.24, label="Pointwise aggregate envelope",
        )
        axes[0].step(
            hours, step_window(P_agg_min), where="post", color="#0072B2",
            linestyle="--", linewidth=1.05, label="Pointwise minimum",
        )
        axes[0].step(
            hours, step_window(P_agg_max), where="post", color="#009E73",
            linewidth=1.05, label="Pointwise maximum",
        )
        axes[0].step(
            hours, step_window(P_agg_FC), where="post", color="#333333",
            linewidth=1.0, label="Forecast baseline",
        )
        axes[0].set(
            title="(a) Pointwise aggregate power flexibility — Type 2 appliances",
            ylabel="Aggregate power (kW)", xlabel="Time of day",
        )
        axes[0].legend(loc="upper center", ncol=2, frameon=True)

        axes[1].fill_between(
            hours, step_window(N_agg_min), step_window(N_agg_max), step="post",
            color="#56B4E9", alpha=0.24,
        )
        axes[1].step(
            hours, step_window(N_agg_min), where="post", color="#0072B2",
            linestyle="--", linewidth=1.05, label="Pointwise minimum",
        )
        axes[1].step(
            hours, step_window(N_agg_max), where="post", color="#009E73",
            linewidth=1.05, label="Pointwise maximum",
        )
        axes[1].step(
            hours, step_window(N_agg_FC), where="post", color="#333333",
            linewidth=1.0, label="Forecast baseline",
        )
        axes[1].set(
            title="(b) Pointwise number of active Type 2 appliances",
            ylabel="Number of active appliances", xlabel="Time of day",
            ylim=(-0.15, len(fq_devices) + 0.5),
        )
        axes[1].set_yticks(np.arange(len(fq_devices) + 1))
        axes[1].legend(loc="upper center", ncol=3, frameon=True)

        for ax in axes:
            ax.set_xlim(time_start / 60.0, time_finish / 60.0)
            ax.set_xticks(tick_hours, tick_labels)
            ax.grid(axis="y", color="0.88", linewidth=0.6)
            ax.set_axisbelow(True)
        axes[0].tick_params(axis="x", labelbottom=True)

    return fig, axes


fig_envelope, axes_envelope = plot_aggregate_flexibility_envelope(
    time_start=0*60,
    time_finish=24 * 60,
    tick_step=120,
)
plt.show()


# Aggregate only Type 2 appliances so that the RT trajectories and the
# flexibility envelope refer to exactly the same population.
P_agg_utility_rt = (
    X_utility_rt_matrix[:, fq_devices] @ fq_power_W / 1000.0
)
P_agg_energy_rt = (
    X_energy_rt_matrix[:, fq_devices] @ fq_power_W / 1000.0
)
N_agg_utility_rt = X_utility_rt_matrix[:, fq_devices].sum(axis=1)
N_agg_energy_rt = X_energy_rt_matrix[:, fq_devices].sum(axis=1)


def plot_rt_trajectories_in_flexibility_envelope(
    time_start=0,
    time_finish=24 * 60,
    tick_step=120,
):
    """Plot the two RT Type 2 trajectories inside the pointwise envelope."""
    hours, tick_hours, tick_labels, step_window = _fq_plot_window(
        time_start, time_finish, tick_step
    )

    with plt.rc_context({
        "font.size": 10,
        "axes.titlesize": 11,
        "axes.labelsize": 10,
        "legend.fontsize": 9,
        "axes.spines.top": False,
        "axes.spines.right": False,
    }):
        fig, axes = plt.subplots(
            2, 1, figsize=(12, 6.8), sharex=True,
            gridspec_kw={"height_ratios": [1.5, 1]}, constrained_layout=True,
        )

        axes[0].fill_between(
            hours,
            step_window(P_agg_min),
            step_window(P_agg_max),
            step="post",
            color="#B9DDF1",
            alpha=0.35,
            label="Pointwise aggregate envelope",
        )
        axes[0].step(
            hours, step_window(P_agg_FC), where="post",
            color="#333333", linewidth=1.0, label="Forecast baseline",
        )
        axes[0].step(
            hours, step_window(P_agg_utility_rt), where="post",
            color="#E69F00", linewidth=1.15, label="RT Max Utility",
        )
        axes[0].step(
            hours, step_window(P_agg_energy_rt), where="post",
            color="#CC79A7", linewidth=1.15, label="RT Min Consumption",
        )
        axes[0].set(
            title="(a) Real-time Type 2 power trajectories within the flexibility envelope",
            ylabel="Aggregate power (kW)",
            xlabel="Time of day",
        )
        axes[0].legend(loc="upper center", ncol=2, frameon=True)

        axes[1].fill_between(
            hours,
            step_window(N_agg_min),
            step_window(N_agg_max),
            step="post",
            color="#B9DDF1",
            alpha=0.35,
        )
        axes[1].step(
            hours, step_window(N_agg_FC), where="post",
            color="#333333", linewidth=1.0, label="Forecast baseline",
        )
        axes[1].step(
            hours, step_window(N_agg_utility_rt), where="post",
            color="#E69F00", linewidth=1.15, label="RT Max Utility",
        )
        axes[1].step(
            hours, step_window(N_agg_energy_rt), where="post",
            color="#CC79A7", linewidth=1.15, label="RT Min Consumption",
        )
        axes[1].set(
            title="(b) Active Type 2 appliances under real-time control",
            ylabel="Number of active appliances",
            xlabel="Time of day",
            ylim=(-0.15, len(fq_devices) + 0.5),
        )
        axes[1].set_yticks(np.arange(len(fq_devices) + 1))
        axes[1].legend(loc="upper center", ncol=3, frameon=True)

        for ax in axes:
            ax.set_xlim(time_start / 60.0, time_finish / 60.0)
            ax.set_xticks(tick_hours, tick_labels)
            ax.grid(axis="y", color="0.88", linewidth=0.6)
            ax.set_axisbelow(True)
        axes[0].tick_params(axis="x", labelbottom=True)

    return fig, axes


fig_rt_envelope, axes_rt_envelope = (
    plot_rt_trajectories_in_flexibility_envelope(
        time_start=0 * 60,
        time_finish=1 * 60,
        tick_step=120,
    )
)
plt.show()


## Complete OFF-Priority and ON-Priority Trajectories

The following figure places two complete admissible trajectories inside the pointwise envelope:

- **OFF-priority:** choose OFF whenever the existing operational rules permit it;
- **ON-priority:** choose ON whenever the existing operational rules permit it.

Each trajectory maintains its own weight history. The curves can cross because prolonged OFF operation raises the OFF-priority weight until activation becomes compulsory, while prolonged ON operation lowers the ON-priority weight until deactivation becomes compulsory. Their crossing is therefore a consequence of temporal coupling, not an error in the pointwise envelope.

In [ ]:
def fq_priority_trajectory(device, preferred_state):
    problem = _fq_device_setup(device)
    trajectory = np.asarray(X_onoff_FC_matrix[:, device], dtype=np.int8).copy()
    if problem is not None:
        local_path, _ = _fq_path(problem, preference=preferred_state)
        if local_path is None:
            raise RuntimeError(f"No priority trajectory for {dev_names[device]}.")
        trajectory[problem["start"]:problem["stop"]] = local_path
    failures, _ = fq_replay(device, trajectory)
    if failures:
        raise AssertionError(f"Invalid priority trajectory: {failures[0]}")
    return trajectory


X_off_priority = np.column_stack([
    fq_priority_trajectory(int(j), preferred_state=0) for j in fq_devices
])
X_on_priority = np.column_stack([
    fq_priority_trajectory(int(j), preferred_state=1) for j in fq_devices
])
P_off_priority = X_off_priority @ fq_power_W / 1000.0
P_on_priority = X_on_priority @ fq_power_W / 1000.0
N_off_priority = X_off_priority.sum(axis=1)
N_on_priority = X_on_priority.sum(axis=1)

for series in (P_off_priority, P_on_priority):
    if not np.all((series >= P_agg_min - 1e-10) &
                  (series <= P_agg_max + 1e-10)):
        raise AssertionError("A priority trajectory lies outside the envelope.")

priority_trajectory_summary = pd.DataFrame({
    "Trajectory": ["OFF-priority", "ON-priority"],
    "Daily Type 2 energy [kWh]": [
        P_off_priority.sum() / 60.0,
        P_on_priority.sum() / 60.0,
    ],
    "Mean active Type 2 appliances": [
        N_off_priority.mean(), N_on_priority.mean(),
    ],
})
display(priority_trajectory_summary.style.hide(axis="index").format({
    "Daily Type 2 energy [kWh]": "{:.3f}",
    "Mean active Type 2 appliances": "{:.3f}",
}))

def plot_priority_trajectories(
    time_start=0,
    time_finish=24 * 60,
    tick_step=120,
):
    """Plot complete OFF/ON-priority trajectories over a selected interval."""
    hours, tick_hours, tick_labels, step_window = _fq_plot_window(
        time_start, time_finish, tick_step
    )

    with plt.rc_context({
        "font.size": 10,
        "axes.titlesize": 11,
        "axes.labelsize": 10,
        "legend.fontsize": 9,
        "axes.spines.top": False,
        "axes.spines.right": False,
    }):
        fig, axes = plt.subplots(
            2, 1, figsize=(12, 6.8), sharex=True,
            gridspec_kw={"height_ratios": [1.5, 1]}, constrained_layout=True,
        )
        axes[0].fill_between(
            hours, step_window(P_agg_min), step_window(P_agg_max), step="post",
            color="#B9DDF1", alpha=0.30, label="Pointwise aggregate envelope",
        )
        axes[0].step(
            hours, step_window(P_agg_FC), where="post", color="#555555",
            linewidth=0.9, alpha=0.8, label="Forecast baseline",
        )
        axes[0].step(
            hours, step_window(P_off_priority), where="post", color="#D55E00",
            linewidth=1.25, label="OFF-priority trajectory",
        )
        axes[0].step(
            hours, step_window(P_on_priority), where="post", color="#CC79A7",
            linewidth=1.25, label="ON-priority trajectory",
        )
        axes[0].set(
            title="(a) Complete priority trajectories within the aggregate envelope",
            ylabel="Aggregate power (kW)", xlabel="Time of day",
        )
        axes[0].legend(loc="upper center", ncol=2, frameon=True)

        axes[1].fill_between(
            hours, step_window(N_agg_min), step_window(N_agg_max), step="post",
            color="#B9DDF1", alpha=0.30,
        )
        axes[1].step(
            hours, step_window(N_agg_FC), where="post", color="#555555",
            linewidth=0.9, alpha=0.8, label="Forecast baseline",
        )
        axes[1].step(
            hours, step_window(N_off_priority), where="post", color="#D55E00",
            linewidth=1.25, label="OFF-priority trajectory",
        )
        axes[1].step(
            hours, step_window(N_on_priority), where="post", color="#CC79A7",
            linewidth=1.25, label="ON-priority trajectory",
        )
        axes[1].set(
            title="(b) Active appliances under the complete priority trajectories",
            ylabel="Number of active appliances", xlabel="Time of day",
            ylim=(-0.15, len(fq_devices) + 0.5),
        )
        axes[1].set_yticks(np.arange(len(fq_devices) + 1))
        axes[1].legend(loc="upper center", ncol=3, frameon=True)

        for ax in axes:
            ax.set_xlim(time_start / 60.0, time_finish / 60.0)
            ax.set_xticks(tick_hours, tick_labels)
            ax.grid(axis="y", color="0.88", linewidth=0.6)
            ax.set_axisbelow(True)
        axes[0].tick_params(axis="x", labelbottom=True)

    return fig, axes


fig_priority, axes_priority = plot_priority_trajectories(
    time_start=18 * 60,
    time_finish=19 * 60,
    tick_step=15,
)
plt.show()

priority_crossings = pd.DataFrame({
    "Comparison": ["Power", "Active-appliance count"],
    "Minutes OFF-priority > ON-priority": [
        int((P_off_priority > P_on_priority + 1e-10).sum()),
        int((N_off_priority > N_on_priority).sum()),
    ],
})
display(priority_crossings.style.hide(axis="index"))


## Instantaneous Flexibility Conditioned on Each Trajectory

Each figure fixes the executed history of its own scenario up to the start of each minute. The lower and upper limits represent the admissible decisions for that minute, using that scenario's **pre-decision weights**. The central curve is the scenario's executed power/count. Only `Type == 2` appliances are included.

The forecast, Max Utility, and Min Energy histories are evaluated separately with the existing RT initialization, operating windows, `force_on`, `force_off`, and `prevent_off` rules. The current RT model uses reactive thresholds without clipping; no new `prevent_on` or hard weight bounds are added. The reference used by `prevent_off` remains the forecast, as in the RT controller.

These are appliance-level instantaneous bounds, without a kitchen power limit. They do not indicate that all intermediate binary aggregate powers are attainable, or that the boundary curves can be executed consecutively. An alternative action changes future weights, so future bounds would have to be recalculated. These figures differ from the geometric envelope above, which allows alternative histories before the selected minute.


The blue shaded range is bounded by minimum and maximum admissible power/counts. The darker scenario profile is the forecast or executed decision. The vertical gap below the power profile represents available downward flexibility; the gap above represents available upward flexibility. A profile coinciding with the lower boundary has no remaining downward flexibility at that minute. One shared legend is used for both panels.

The baseline figure covers the full day. RT figures focus on 17:00–19:00. The bounds are evaluated before the current decision, whereas the reference curve is the executed decision. Min Energy selecting the lower bound is therefore an observation about immediate Type 2 consumption, not a guarantee of minimum daily energy or lower future bounds. The earlier downward-flexibility table includes Types 1 and 2 before optimization and measures a different quantity.

In [ ]:
def compute_trajectory_conditioned_flexibility(X_trajectory, W_trajectory, name,
                                             Y_trajectory, power_limit_W):
    """Replay an executed history; evaluate both actions at its pre-decision state."""
    states = np.asarray(X_trajectory)
    expected_weights = np.asarray(W_trajectory)
    if states.shape != (T, N_DEV) or expected_weights.shape != (T, N_DEV):
        raise ValueError("Trajectory arrays must have shape (T, N_DEV).")
    if not np.all(np.isin(states, (0, 1))):
        raise ValueError("Trajectory states must be binary.")
    devices = np.flatnonzero(np.asarray(Type) == 2)
    xmin = states[:, devices].copy()
    xmax = states[:, devices].copy()
    weights = np.full((T, N_DEV), np.nan)
    weights_before = np.full((T, N_DEV), np.nan)
    previous = initialize_time_weights_from_state(
        X0=X_onoff_FC_matrix[0], Type=Type,
        T_ON_0=T_ON_FC_matrix[0], T_OFF_0=T_OFF_FC_matrix[0],
        time_flex_available=time_flex_available_matrix[0],
    )
    for minute in range(T):
        previous = prepare_time_weights_for_minute(
            prev_W_T=previous, X_reference=X_plan_ref_matrix[minute], Type=Type,
            time_flex_available=time_flex_available_matrix[minute],
            time_flex_start=time_flex_start_matrix[minute],
        )
        weights_before[minute] = previous
        before, force_on, force_off = function_dataset(
            i=minute, X_RT_matrix=states, W_T_RT_matrix=weights,
            X_plan_matrix=X_plan_ref_matrix, Type=Type,
            time_flex_available_matrix=time_flex_available_matrix,
            time_flex_start_matrix=time_flex_start_matrix,
            variable_hysteresis=False,
        )
        for col, j in enumerate(devices):
            if not time_flex_available_matrix[minute, j]:
                xmin[minute, col] = xmax[minute, col] = X_plan_ref_matrix[minute, j]
                continue
            prevent_off = (
                before[j] == 1 and not force_on[j] and not force_off[j]
                and previous[j] + 9.0 / T_OFF_FC_matrix[minute, j]
                >= get_w_max(j, minute, False) - 1e-9
            )
            xmin[minute, col] = int(force_on[j] or prevent_off)
            xmax[minute, col] = int(not force_off[j])
        previous = update_time_weights(
            prev_W_T=previous, X_now=states[minute], Type=Type,
            T_ON_now=T_ON_FC_matrix[minute], T_OFF_now=T_OFF_FC_matrix[minute],
            minute=minute, time_flex_available=time_flex_available_matrix[minute],
            variable_hysteresis=False,
        )
        weights[minute] = previous
    if not np.allclose(weights[:, devices], expected_weights[:, devices],
                       rtol=0, atol=1e-10, equal_nan=True):
        raise ValueError(f"{name}: replayed weights differ from the stored trajectory.")
    actual = states[:, devices]
    bad = np.argwhere((actual < xmin) | (actual > xmax))
    if len(bad):
        minute, col = bad[0]
        raise ValueError(f"{name}: executed state is not admissible at minute {minute}, "
                         f"device {dev_names[devices[col]]}. Check RT rule settings.")
    power = np.asarray(Power_max)[devices] / 1000.0
    frame = pd.DataFrame({
        'Minute': np.arange(T),
        'P_min [kW]': xmin @ power, 'P_reference [kW]': actual @ power,
        'P_max [kW]': xmax @ power,
        'N_min': xmin.sum(axis=1), 'N_reference': actual.sum(axis=1),
        'N_max': xmax.sum(axis=1),
    })
    frame['Downward [kW]'] = frame['P_reference [kW]'] - frame['P_min [kW]']
    frame['Upward [kW]'] = frame['P_max [kW]'] - frame['P_reference [kW]']
    # Hold the other appliances at their executed power in this scenario.
    # This is the available kitchen capacity for Type 2, not an additional
    # appliance constraint and not the full kitchen power limit.
    other = np.asarray(Type) != 2
    fractions = np.asarray(Y_trajectory, dtype=float)
    limits = np.asarray(power_limit_W, dtype=float)
    if fractions.shape != states.shape or limits.shape != (T,):
        raise ValueError('Y_trajectory or power_limit_W has an invalid shape.')
    if not np.isfinite(fractions).all() or not np.isfinite(limits).all():
        raise ValueError('Power fractions and limits must be finite.')
    other_power = (states[:, other] * fractions[:, other]
                   * np.asarray(Power_max)[other]).sum(axis=1) / 1000.0
    frame['Other appliances [kW]'] = other_power
    frame['Kitchen power limit [kW]'] = limits / 1000.0
    frame['Type 2 power budget [kW]'] = limits / 1000.0 - other_power
    frame['Minimum excess [kW]'] = np.maximum(
        frame['P_min [kW]'] - frame['Type 2 power budget [kW]'], 0)
    frame['Executed excess [kW]'] = np.maximum(
        frame['P_reference [kW]'] - frame['Type 2 power budget [kW]'], 0)
    # Do not clip a negative budget: it means the other loads alone exceed
    # the kitchen limit, even if every Type 2 appliance were OFF.
    frame.attrs['scenario'] = name
    return frame, weights_before[:, devices]


def plot_trajectory_conditioned_flexibility(metrics, time_start=0,
                                           time_finish=24 * 60, tick_step=120):
    """Plot instantaneous bounds for one history; finish is exclusive, in minutes."""
    if any(not isinstance(v, (int, np.integer)) for v in (time_start, time_finish, tick_step)):
        raise ValueError('Time arguments must be integer minutes.')
    if not (0 <= time_start < time_finish <= len(metrics)) or tick_step <= 0:
        raise ValueError('Invalid interval or tick_step.')
    window = metrics.iloc[time_start:time_finish]
    hours = np.arange(time_start, time_finish + 1) / 60.0
    def step(column):
        values = window[column].to_numpy()
        return np.r_[values, values[-1]]
    ticks = np.arange(time_start, time_finish, tick_step)
    ticks = np.r_[ticks, time_finish]
    name = metrics.attrs['scenario']
    reference_color = {'Baseline': '#333333', 'RT Max Utility': '#D55E00',
                       'RT Min Energy': '#CC79A7'}.get(name, '#333333')
    with plt.rc_context({'font.size': 10, 'axes.titlesize': 12, 'legend.fontsize': 10,
                         'axes.spines.top': False, 'axes.spines.right': False}):
        fig, axes = plt.subplots(2, 1, figsize=(12, 6.8), sharex=True,
                                 gridspec_kw={'height_ratios': [1.5, 1]},
                                 constrained_layout=False)
        fig.suptitle(
            f'{name} — instantaneous flexibility of time-flexible appliances',
            y=.99, fontsize=16, fontweight='bold',
        )
        for ax, columns, title, ylabel in (
            (axes[0], ('P_min [kW]', 'P_reference [kW]', 'P_max [kW]'),
             '(a) Aggregate power', 'Aggregate power (kW)'),
            (axes[1], ('N_min', 'N_reference', 'N_max'),
             '(b) Active appliances', 'Nº active appliances'),
        ):
            low, reference, high = columns
            ax.fill_between(hours, step(low), step(high), step='post',
                            color='#56B4E9', alpha=.20, label='Admissible range')
            ax.step(hours, step(low), where='post', color='#0072B2',
                    linestyle='--', linewidth=1.05, label='Minimum admissible')
            ax.step(hours, step(high), where='post', color='#009E73',
                    linewidth=1.05, label='Maximum admissible')
            ax.step(hours, step(reference), where='post', color=reference_color,
                    linewidth=1.45, label='Forecast profile' if name == 'Baseline' else 'Executed profile', zorder=4)
            ax.set(title=title, ylabel=ylabel, xlabel='Time of day' if ax is axes[1] else '',
                   xlim=(time_start / 60, time_finish / 60))
            ax.set_xticks(ticks / 60, [f'{t//60:02d}:{t%60:02d}' for t in ticks])
            ax.grid(axis='y', color='0.88', linewidth=.6)
            ax.set_axisbelow(True)
            ax.title.set_fontweight('bold')
            ax.yaxis.label.set_fontsize(12)
            ax.yaxis.label.set_fontweight('bold')
            ax.xaxis.label.set_fontsize(12)
            ax.xaxis.label.set_fontweight('bold')
        # Highlight each complete violating minute on the executed profile.
        # Keep the full-day baseline free of violation markers.
        if name != 'Baseline':
            violation = window['Executed excess [kW]'].to_numpy() > 1e-9
            axes[0].hlines(
                window['P_reference [kW]'].to_numpy()[violation],
                hours[:-1][violation], hours[1:][violation],
                colors='#C00000', linewidths=2.6, zorder=6,
                label='Kitchen power-limit violation',
            )
        ymax = max(window['P_max [kW]'].max(), .1)
        axes[0].set_ylim(-.02 * ymax, 1.07 * ymax)
        n_devices = int(np.sum(np.asarray(Type) == 2))
        axes[1].set_ylim(-.15, n_devices + .35)
        axes[1].set_yticks(np.arange(n_devices + 1))
        handles, labels = axes[0].get_legend_handles_labels()
        fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(.5, .93),
                   ncol=4, frameon=False, fontsize=10)
        fig.tight_layout(rect=(0, .02, 1, .86))
    return fig, axes


conditioned_flexibility = {}
conditioned_weights_before = {}
for scenario, states, stored_weights, fractions, limits in (
    ('Baseline', X_onoff_FC_matrix, WT_FC_matrix, Y_FC_matrix,
     np.array([get_p_lim_for_minute(t) for t in range(T)])),
    ('RT Max Utility', X_utility_rt_matrix, W_T_utility_rt_matrix,
     Y_utility_rt_matrix, P_lim_utility_vector),
    ('RT Min Energy', X_energy_rt_matrix, W_T_energy_rt_matrix,
     Y_energy_rt_matrix, P_lim_energy_vector),
):
    conditioned_flexibility[scenario], conditioned_weights_before[scenario] = (
        compute_trajectory_conditioned_flexibility(states, stored_weights, scenario, fractions, limits)
    )
print('All three histories replayed successfully; executed states lie within their own bounds.')


In [ ]:
fig_conditioned_baseline, axes_conditioned_baseline = plot_trajectory_conditioned_flexibility(
    conditioned_flexibility['Baseline'],
    time_start=0,
    time_finish=24 * 60,
    tick_step=120,
)
plt.show()

In [ ]:
fig_conditioned_utility, axes_conditioned_utility = plot_trajectory_conditioned_flexibility(
    conditioned_flexibility['RT Max Utility'],
    time_start=17*60,
    time_finish=19 * 60,
    tick_step=15,
)
plt.show()

In [ ]:
fig_conditioned_energy, axes_conditioned_energy = plot_trajectory_conditioned_flexibility(
    conditioned_flexibility['RT Min Energy'],
    time_start=17*60,
    time_finish=19 * 60,
    tick_step=15,
)
plt.show()

In [ ]:
# Compare the same interval without assuming that one minimum is always higher.
conditioned_comparison_start = 17 * 60
conditioned_comparison_finish = 19 * 60
rows = []
baseline_min = conditioned_flexibility['Baseline']['P_min [kW]'].iloc[
    conditioned_comparison_start:conditioned_comparison_finish].to_numpy()
for name, data in conditioned_flexibility.items():
    w = data.iloc[conditioned_comparison_start:conditioned_comparison_finish]
    rows.append({
        'Scenario': name,
        'Mean minimum Type 2 power [kW]': w['P_min [kW]'].mean(),
        'Minutes at Type 2 minimum': int(np.isclose(w['P_reference [kW]'], w['P_min [kW]'], rtol=0, atol=1e-9).sum()),
        'Minutes minimum above baseline minimum': int((w['P_min [kW]'].to_numpy() > baseline_min + 1e-9).sum()),
    })
conditioned_comparison = pd.DataFrame(rows)
display(conditioned_comparison.style.hide(axis='index').format({'Mean minimum Type 2 power [kW]': '{:.3f}'}))


In [ ]:
def plot_conditioned_flexibility_comparison(
    scenarios, time_start=17 * 60, time_finish=18 * 60, tick_step=15,
):
    """Compare scenario-conditioned Type 2 power envelopes side by side."""
    if not (0 <= time_start < time_finish <= T):
        raise ValueError('Expected 0 <= time_start < time_finish <= T.')
    if tick_step <= 0:
        raise ValueError('tick_step must be positive.')

    colors = {
        'Baseline': '#333333',
        'RT Max Utility': '#D55E00',
        'RT Min Energy': '#CC79A7',
    }
    titles = {
        'Baseline': 'Baseline',
        'RT Max Utility': 'Max Utility',
        'RT Min Energy': 'Min Consumption',
    }
    minutes = np.arange(time_start, time_finish + 1)
    hours = minutes / 60.0
    ticks = np.arange(time_start, time_finish, tick_step, dtype=int)
    ticks = np.r_[ticks, time_finish]

    with plt.rc_context({
        'font.size': 11, 'axes.titlesize': 13, 'legend.fontsize': 11,
        'axes.spines.top': False, 'axes.spines.right': False,
    }):
        fig, axes = plt.subplots(
            1, len(scenarios), figsize=(16, 5.0), sharex=True, sharey=True,
            constrained_layout=False,
        )
        axes = np.atleast_1d(axes)
        for ax, name in zip(axes, scenarios):
            window = conditioned_flexibility[name].iloc[time_start:time_finish]

            def step(column):
                values = window[column].to_numpy()
                return np.r_[values, values[-1]]

            ax.fill_between(
                hours, step('P_min [kW]'), step('P_max [kW]'), step='post',
                color='#56B4E9', alpha=.20, label='Admissible range',
            )
            ax.step(
                hours, step('P_min [kW]'), where='post', color='#0072B2',
                linestyle='--', linewidth=1.05, label='Minimum admissible',
            )
            ax.step(
                hours, step('P_max [kW]'), where='post', color='#009E73',
                linewidth=1.05, label='Maximum admissible',
            )
            ax.step(
                hours, step('P_reference [kW]'), where='post',
                color=colors[name], linewidth=1.5,
                label='Forecast profile' if name == 'Baseline' else '_nolegend_',
                zorder=4,
            )
            if name != 'Baseline':
                violation = window['Executed excess [kW]'].to_numpy() > 1e-9
                ax.hlines(
                    window['P_reference [kW]'].to_numpy()[violation],
                    hours[:-1][violation], hours[1:][violation],
                    colors='#C00000', linewidths=2.8, zorder=6,
                    label='Kitchen power-limit violation',
                )
            ax.set_title(titles.get(name, name), fontweight='bold')
            ax.set_xlabel('Time of day')
            ax.set_xlim(time_start / 60.0, time_finish / 60.0)
            ax.set_xticks(
                ticks / 60.0,
                [f'{t // 60:02d}:{t % 60:02d}' for t in ticks],
            )
            ax.grid(axis='y', color='0.88', linewidth=.6)
            ax.set_axisbelow(True)
            ax.xaxis.label.set_fontsize(13)
            ax.xaxis.label.set_fontweight('bold')

        axes[0].set_ylabel(
            'Aggregate Type 2 power (kW)', fontsize=13, fontweight='bold',
        )
        ymax = max(
            conditioned_flexibility[name]['P_max [kW]'].iloc[time_start:time_finish].max()
            for name in scenarios
        )
        axes[0].set_ylim(-.02 * ymax, 1.07 * ymax)
        handles, labels = [], []
        for ax in axes:
            for handle, label in zip(*ax.get_legend_handles_labels()):
                if label not in labels:
                    handles.append(handle)
                    labels.append(label)
        fig.legend(
            handles, labels, loc='upper center', bbox_to_anchor=(.5, .98),
            ncol=5, frameon=False, fontsize=11,
        )
        fig.suptitle(
            'Instantaneous flexibility of time-flexible appliances',
            y=1.08, fontsize=18, fontweight='bold',
        )
        fig.tight_layout(rect=(0, .02, 1, .86), w_pad=1.2)
    return fig, axes


fig_conditioned_comparison, axes_conditioned_comparison = (
    plot_conditioned_flexibility_comparison(
        ['Baseline', 'RT Max Utility', 'RT Min Energy'],
        time_start=17 * 60,
        time_finish=18 * 60,
        tick_step=15,
    )
)
plt.show()


## Exhaust power-flexibility weights

Preserve the Exhaust view used by the thesis figure `RT_Exhaust_Weights.pdf`.

In [ ]:
fig, ax = plot_power_flexible_Y_comparison(
    device="Exhaust",  # Ou "Light2", "Exhaust", 10, 11, 12
    dev_names=dev_names,
    Type=Type,
    X_baseline=X_onoff_FC_matrix,
    Y_baseline=Y_FC_matrix,
    X_utility=X_utility_rt_matrix,
    Y_utility=Y_utility_rt_matrix,
    X_energy=X_energy_rt_matrix,
    Y_energy=Y_energy_rt_matrix,
    time_start=00 * 60,
    time_finish=24 * 60,
    tick_step=60,
)